<a href="https://colab.research.google.com/github/cinarersoz53-glitch/dgjb/blob/main/siberDil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
print("Türkçe Oltalama Mesajı Tespit Projesi")
print("Siber güvenlik projemize hoş geldiniz!")

Türkçe Oltalama Mesajı Tespit Projesi
Siber güvenlik projemize hoş geldiniz!


In [4]:
import pandas as pd
import re
import string

print("Gerekli kütüphaneler hazır!")

Gerekli kütüphaneler hazır!


In [5]:
veriler = [
    {
        "mesaj": "ACİL! Hesabınız kapatılacak, hemen bağlantıya tıklayın!",
        "etiket": 1
    },
    {
        "mesaj": "Tebrikler ödül kazandınız, ödülünüzü almak için hemen işlem yapın!",
        "etiket": 1
    },
    {
        "mesaj": "Merhaba, yarın saat 10'da toplantımız var.",
        "etiket": 0
    },
    {
        "mesaj": "Ödev dosyasını sana e-posta ile gönderdim.",
        "etiket": 0
    }
]

print("Örnek mesaj sayısı:", len(veriler))

Örnek mesaj sayısı: 4


In [6]:
df = pd.DataFrame(veriler)

display(df)

,mesaj,etiket
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1
2,"Merhaba, yarın saat 10'da toplantımız var.",0
3,Ödev dosyasını sana e-posta ile gönderdim.,0


In [7]:
df["kelime_sayisi"] = df["mesaj"].apply(
    lambda mesaj: len(mesaj.split())
)

display(df)

,mesaj,etiket,kelime_sayisi
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,6
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,9
2,"Merhaba, yarın saat 10'da toplantımız var.",0,6
3,Ödev dosyasını sana e-posta ile gönderdim.,0,6


In [8]:
acil_kelimeler = [
    "acil", "hemen", "derhal", "son", "askıya",
    "kapatılacak", "tıklayın", "ödül", "tebrikler"
]

def acil_kelime_sayisi(mesaj):
    kelimeler = re.findall(r"\w+", mesaj.lower())
    return sum(1 for kelime in kelimeler if kelime in acil_kelimeler)

df["acil_kelime_sayisi"] = df["mesaj"].apply(acil_kelime_sayisi)

display(df)

,mesaj,etiket,kelime_sayisi,acil_kelime_sayisi
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,6,3
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,9,3
2,"Merhaba, yarın saat 10'da toplantımız var.",0,6,0
3,Ödev dosyasını sana e-posta ile gönderdim.,0,6,0


In [9]:

import re

acil_kelimeler = [
    "acil", "hemen", "derhal", "son", "askıya",
    "kapatılacak", "tıklayın", "ödül", "tebrikler"
]

def turkce_kucult(metin):
    metin = metin.replace("İ", "i").replace("I", "ı")
    return metin.lower()

def acil_kelime_sayisi(mesaj):
    mesaj = turkce_kucult(mesaj)
    kelimeler = re.findall(r"\w+", mesaj)
    return sum(1 for kelime in kelimeler if kelime in acil_kelimeler)

df["acil_kelime_sayisi"] = df["mesaj"].apply(acil_kelime_sayisi)

display(df)


,mesaj,etiket,kelime_sayisi,acil_kelime_sayisi
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,6,4
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,9,3
2,"Merhaba, yarın saat 10'da toplantımız var.",0,6,0
3,Ödev dosyasını sana e-posta ile gönderdim.,0,6,0


In [10]:

import re

baski_kaliplari = {
    "sure_baskisi": [
        r"\d+\s*saat\s*(içinde|içerisinde)",
        r"son\s+\d+\s*(saat|gün)",
        r"vaktiniz\s*(doluyor|dolmak üzere)"
    ],
    "hesap_tehdidi": [
        r"hesabınız\s*(askıya alınacak|kapatılacak)",
        r"hesabınız\s*askıya\s*alınmıştır",
        r"erişiminiz\s*(engellenecek|kısıtlanacak)"
    ],
    "acil_eylem": [
        r"hemen\s*(tıklayın|doğrulayın|giriş yapın)",
        r"derhal\s*(işlem yapın|onaylayın)",
        r"son\s*şansınız"
    ]
}

def baski_analizi(mesaj):
    mesaj = turkce_kucult(mesaj)
    sonuc = {}

    for kategori, kaliplar in baski_kaliplari.items():
        sonuc[kategori] = int(
            any(re.search(kalip, mesaj) for kalip in kaliplar)
        )

    return sonuc

analiz = df["mesaj"].apply(baski_analizi)

baski_df = analiz.apply(lambda x: __import__("pandas").Series(x))

df = __import__("pandas").concat([df, baski_df], axis=1)

display(df)


,mesaj,etiket,kelime_sayisi,acil_kelime_sayisi,sure_baskisi,hesap_tehdidi,acil_eylem
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,6,4,0,1,0
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,9,3,0,0,0
2,"Merhaba, yarın saat 10'da toplantımız var.",0,6,0,0,0,0
3,Ödev dosyasını sana e-posta ile gönderdim.,0,6,0,0,0,0


In [11]:

# Acil eylem kalıplarını güncelle
baski_kaliplari["acil_eylem"] = [
    r"\bhemen\b(?:\s+\w+){0,3}\s+(?:tıklayın|doğrulayın|onaylayın)",
    r"\bderhal\b(?:\s+\w+){0,3}\s+(?:işlem yapın|onaylayın)",
    r"\bson\s+şansınız\b",
    r"\bgecikmeden\b(?:\s+\w+){0,3}\s+(?:tıklayın|doğrulayın)"
]

# Güncellenmiş kurallarla yeniden analiz et
analiz = df["mesaj"].apply(baski_analizi)

# Sonuçları mevcut sütunlara yaz
for kategori in baski_kaliplari:
    df[kategori] = analiz.apply(lambda x: x[kategori])

# Sonuçları görüntüle
display(df[[
    "mesaj",
    "sure_baskisi",
    "hesap_tehdidi",
    "acil_eylem"
]])


,mesaj,sure_baskisi,hesap_tehdidi,acil_eylem
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",0,1,1
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",0,0,0
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0,0
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0,0


In [12]:

# Ödül ve hediye vaadi kalıpları
odul_kaliplari = [
    r"\bödül\s+kazandınız\b",
    r"\bhediye\s+kazandınız\b",
    r"\btebrikler\b.{0,60}\bkazandınız\b",
    r"\bçekiliş(?:i|te|ten)?\s+kazandınız\b",
    r"\bödülünüzü\s+almak\b",
    r"\bhediyenizi\s+almak\b",
    r"\bücretsiz\s+hediye\b",
    r"\bpara\s+ödülü\b"
]

def odul_vaadi_analizi(mesaj):
    mesaj = turkce_kucult(mesaj)

    for kalip in odul_kaliplari:
        if re.search(kalip, mesaj):
            return 1

    return 0

# Yeni özelliği veri setine ekle
df["odul_vaadi"] = df["mesaj"].apply(odul_vaadi_analizi)

# Sonuçları görüntüle
display(df[[
    "mesaj",
    "sure_baskisi",
    "hesap_tehdidi",
    "acil_eylem",
    "odul_vaadi"
]])


,mesaj,sure_baskisi,hesap_tehdidi,acil_eylem,odul_vaadi
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",0,1,1,0
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",0,0,0,1
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0,0,0
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0,0,0


In [13]:

# ADIM 4: Hassas bilgi talebi analizi

import re

# Hassas bilgi türleri
hassas_bilgiler = [
    r"şifre(?:niz|nizi|nizi)?",
    r"parola(?:nız|nızı)?",
    r"t\.?\s*c\.?\s*kimlik\s*(?:no|numarası|numaranız)",
    r"kart\s*(?:numarası|bilgileri|bilgilerinizi)",
    r"cvv",
    r"doğrulama\s*kod(?:u|unu|unuzu)",
    r"sms\s*kod(?:u|unu|unuzu)"
]

# Bilgi isteme ifadeleri
talep_ifadeleri = [
    r"gir(?:in|iniz)",
    r"paylaş(?:ın|ınız)",
    r"gönder(?:in|iniz)",
    r"bildir(?:in|iniz)",
    r"yaz(?:ın|ınız)",
    r"ilet(?:in|iniz)",
    r"doğrula(?:yın|yınız)",
    r"güncelle(?:yin|yiniz)"
]

def bilgi_talebi_analizi(mesaj):
    mesaj = turkce_kucult(mesaj)

    hassas_var = any(
        re.search(kalip, mesaj)
        for kalip in hassas_bilgiler
    )

    talep_var = any(
        re.search(kalip, mesaj)
        for kalip in talep_ifadeleri
    )

    return int(hassas_var and talep_var)

# Veri setine yeni özellik ekle
df["bilgi_talebi"] = df["mesaj"].apply(
    bilgi_talebi_analizi
)

display(df[[
    "mesaj",
    "hesap_tehdidi",
    "acil_eylem",
    "odul_vaadi",
    "bilgi_talebi"
]])


,mesaj,hesap_tehdidi,acil_eylem,odul_vaadi,bilgi_talebi
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,1,0,0
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",0,0,1,0
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0,0,0
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0,0,0


In [14]:

test_mesajlari = [
    "Şifrenizi hemen girin.",
    "T.C. kimlik numaranızı paylaşın.",
    "Doğrulama kodunuzu gönderin.",
    "Yarın toplantımız var."
]

for mesaj in test_mesajlari:
    print(mesaj, "→", bilgi_talebi_analizi(mesaj))


Şifrenizi hemen girin. → 1
T.C. kimlik numaranızı paylaşın. → 1
Doğrulama kodunuzu gönderin. → 1
Yarın toplantımız var. → 0


In [15]:

# ADIM 5: URL tespiti

import re

url_kalibi = (
    r"(?<![\w@])"
    r"(?:https?://|www\.)"
    r"[^\s<>\"']+"
)

def url_analizi(mesaj):
    mesaj = str(mesaj)

    bulunan_url = re.findall(
        url_kalibi,
        mesaj,
        flags=re.IGNORECASE
    )

    return int(len(bulunan_url) > 0)

# Yeni özelliği veri setine ekle
df["url_var"] = df["mesaj"].apply(url_analizi)

# Sonuçları göster
display(df[[
    "mesaj",
    "hesap_tehdidi",
    "acil_eylem",
    "odul_vaadi",
    "bilgi_talebi",
    "url_var"
]])

# Algoritmayı farklı mesajlarla test et
test_url_mesajlari = [
    "Hemen https://ornek.com adresine girin.",
    "Ödülünüz için www.ornek.com sitesini ziyaret edin.",
    "Yarın saat 10'da toplantımız var.",
    "Şifrenizi kimseyle paylaşmayın."
]

print("\nURL TEST SONUÇLARI")

for mesaj in test_url_mesajlari:
    print(mesaj, "→", url_analizi(mesaj))


,mesaj,hesap_tehdidi,acil_eylem,odul_vaadi,bilgi_talebi,url_var
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,1,0,0,0
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",0,0,1,0,0
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0,0,0,0
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0,0,0,0



URL TEST SONUÇLARI
Hemen https://ornek.com adresine girin. → 1
Ödülünüz için www.ornek.com sitesini ziyaret edin. → 1
Yarın saat 10'da toplantımız var. → 0
Şifrenizi kimseyle paylaşmayın. → 0


In [16]:

# ADIM 6: Türkçe karakter kullanım analizi

import re

turkce_karakterler = "çğıöşüÇĞİÖŞÜ"

def turkce_karakter_analizi(mesaj):
    mesaj = str(mesaj)

    # Türkçe karakter sayısı
    karakter_sayisi = sum(
        1 for harf in mesaj
        if harf in turkce_karakterler
    )

    # Toplam alfabetik karakter sayısı
    toplam_harf = sum(
        1 for harf in mesaj
        if harf.isalpha()
    )

    # Türkçe karakter oranı
    oran = (
        karakter_sayisi / toplam_harf
        if toplam_harf > 0 else 0
    )

    return round(oran, 4)

# Yeni özelliği veri setine ekle
df["turkce_karakter_orani"] = df["mesaj"].apply(
    turkce_karakter_analizi
)

display(df[[
    "mesaj",
    "etiket",
    "turkce_karakter_orani"
]])

# Test
test_metinleri = [
    "Şifrenizi hemen doğrulayın.",
    "Sifrenizi hemen dogrulayin.",
    "Ödülünüzü almak için tıklayın.",
    "Odulunuzu almak icin tiklayin."
]

print("\nTÜRKÇE KARAKTER ANALİZİ")

for metin in test_metinleri:
    print(
        metin,
        "→",
        turkce_karakter_analizi(metin)
    )


,mesaj,etiket,turkce_karakter_orani
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,0.1702
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,0.2143
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0.0938
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0.1143



TÜRKÇE KARAKTER ANALİZİ
Şifrenizi hemen doğrulayın. → 0.125
Sifrenizi hemen dogrulayin. → 0.0
Ödülünüzü almak için tıklayın. → 0.3077
Odulunuzu almak icin tiklayin. → 0.0


In [17]:

# ADIM 7.1: Türkçe dil analizi altyapısı

import os
import subprocess

# Java sürümünü kontrol et
sonuc = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print("JAVA KONTROLÜ:")
print(sonuc.stderr if sonuc.stderr else sonuc.stdout)

# Zemberek için gerekli Python kütüphanesi
!pip -q install jpype1

import jpype

print("\nJPype başarıyla yüklendi.")
print("JVM çalışıyor mu?", jpype.isJVMStarted())


JAVA KONTROLÜ:
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 7.5 MB/s eta 0:00:00

JPype başarıyla yüklendi.
JVM çalışıyor mu? False


In [18]:

# ADIM 7.3: Zemberek'i başlatma

import jpype
import os

jar_dosyasi = "/content/zemberek-full.jar"

if not os.path.exists(jar_dosyasi):
    print("HATA: Zemberek dosyası bulunamadı.")

else:
    if not jpype.isJVMStarted():
        jpype.startJVM(
            jpype.getDefaultJVMPath(),
            "-ea",
            classpath=[jar_dosyasi],
            convertStrings=True
        )

    TurkishMorphology = jpype.JClass(
        "zemberek.morphology.TurkishMorphology"
    )

    morphology = TurkishMorphology.createWithDefaults()

    print("Zemberek başarıyla başlatıldı!")
    print("JVM çalışıyor:", jpype.isJVMStarted())


Zemberek başarıyla başlatıldı!
JVM çalışıyor: True


In [19]:

# ADIM 7.4: Düzeltilmiş Zemberek kelime analizi

test_kelimeleri = [
    "şifrenizi",
    "sifrenizi",
    "hesabınız",
    "hesabnızı",
    "ödülünüzü",
    "odulunuzu",
    "tıklayın",
    "tiklayin"
]

def kelime_analizi(kelime):
    analizler = morphology.analyze(kelime)
    return analizler.analysisCount() > 0

print("ZEMBEREK KELİME ANALİZİ")
print("-" * 45)

for kelime in test_kelimeleri:
    sonuc = kelime_analizi(kelime)

    if sonuc:
        print(f"{kelime:15} → Tanındı")
    else:
        print(f"{kelime:15} → Tanınmadı")


ZEMBEREK KELİME ANALİZİ
---------------------------------------------
şifrenizi       → Tanındı
sifrenizi       → Tanınmadı
hesabınız       → Tanındı
hesabnızı       → Tanınmadı
ödülünüzü       → Tanındı
odulunuzu       → Tanınmadı
tıklayın        → Tanındı
tiklayin        → Tanınmadı


In [20]:

# ADIM 7.5: Mesaj bazında morfolojik analiz

import re
import pandas as pd

def mesaj_kelime_analizi(mesaj):

    # Türkçe kelimeleri ayır
    kelimeler = re.findall(
        r"[a-zA-ZçğıöşüÇĞİÖŞÜ]+",
        str(mesaj)
    )

    toplam = len(kelimeler)

    if toplam == 0:
        return pd.Series({
            "tanimlanmayan_kelime_sayisi": 0,
            "tanimlanmayan_kelime_orani": 0.0
        })

    taninmayan = 0

    for kelime in kelimeler:
        kelime = turkce_kucult(kelime)

        if not kelime_analizi(kelime):
            taninmayan += 1

    oran = taninmayan / toplam

    return pd.Series({
        "tanimlanmayan_kelime_sayisi": taninmayan,
        "tanimlanmayan_kelime_orani": round(oran, 4)
    })

# Tüm mesajları analiz et
sonuclar = df["mesaj"].apply(mesaj_kelime_analizi)

# Yeni sütunları veri setine ekle
df[
    [
        "tanimlanmayan_kelime_sayisi",
        "tanimlanmayan_kelime_orani"
    ]
] = sonuclar

# Sonuçları görüntüle
display(df[[
    "mesaj",
    "etiket",
    "tanimlanmayan_kelime_sayisi",
    "tanimlanmayan_kelime_orani"
]])


,mesaj,etiket,tanimlanmayan_kelime_sayisi,tanimlanmayan_kelime_orani
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,0.0,0.0
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,0.0,0.0
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0.0,0.0
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0.0,0.0


In [21]:

# ADIM 7.6: Tanınmayan kelimeleri listeleme

import re

def taninmayan_kelimeleri_bul(mesaj):

    kelimeler = re.findall(
        r"[a-zA-ZçğıöşüÇĞİÖŞÜ]+",
        str(mesaj)
    )

    taninmayanlar = []

    for kelime in kelimeler:

        temiz_kelime = turkce_kucult(kelime)

        if not kelime_analizi(temiz_kelime):
            taninmayanlar.append(kelime)

    return taninmayanlar

# Her mesajdaki tanınmayan kelimeleri bul
df["taninmayan_kelimeler"] = df["mesaj"].apply(
    taninmayan_kelimeleri_bul
)

# Sonuçları göster
display(df[[
    "mesaj",
    "etiket",
    "tanimlanmayan_kelime_orani",
    "taninmayan_kelimeler"
]])


,mesaj,etiket,tanimlanmayan_kelime_orani,taninmayan_kelimeler
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,0.0,[]
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,0.0,[]
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0.0,[]
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0.0,[]


In [22]:

# ADIM 7.6.1: Hatalı kelimelerle kontrol testi

test_kelimeleri = [
    "şifrenizi",
    "sifrenizi",
    "hesabınız",
    "hesabnızı",
    "ödülünüzü",
    "odulunuzu",
    "tıklayın",
    "tiklayin",
    "merhaba",
    "merhabaaa",
    "kitap",
    "kitappp"
]

print("ZEMBEREK KONTROL TESTİ")
print("-" * 45)

for kelime in test_kelimeleri:
    analizler = morphology.analyze(kelime)
    sayi = analizler.analysisCount()

    print(
        f"{kelime:15} | "
        f"Analiz sayısı: {sayi}"
    )


ZEMBEREK KONTROL TESTİ
---------------------------------------------
şifrenizi       | Analiz sayısı: 1
sifrenizi       | Analiz sayısı: 0
hesabınız       | Analiz sayısı: 3
hesabnızı       | Analiz sayısı: 0
ödülünüzü       | Analiz sayısı: 1
odulunuzu       | Analiz sayısı: 0
tıklayın        | Analiz sayısı: 1
tiklayin        | Analiz sayısı: 0
merhaba         | Analiz sayısı: 2
merhabaaa       | Analiz sayısı: 0
kitap           | Analiz sayısı: 2
kitappp         | Analiz sayısı: 0


In [23]:

# ADIM 7.7: Hatalı mesajlarla Zemberek testi

test_mesajlari = [
    "Şifrenizi hemen doğrulayın.",
    "Sifrenizi hemen dogrulayin.",
    "Hesabınız kapatılacak.",
    "Hesabnızı kapatılcak.",
    "Ödülünüzü almak için tıklayın.",
    "Odulunuzu almak icin tiklayin.",
    "Merhaba, yarın toplantımız var.",
    "Merhabaaa, yarın toplntımız var."
]

print("MESAJ BAZINDA YAZIM ANALİZİ")
print("-" * 55)

for mesaj in test_mesajlari:

    taninmayanlar = taninmayan_kelimeleri_bul(mesaj)
    sonuc = mesaj_kelime_analizi(mesaj)

    print("\nMesaj:", mesaj)
    print("Tanınmayan kelimeler:", taninmayanlar)
    print(
        "Tanınmayan kelime oranı:",
        sonuc["tanimlanmayan_kelime_orani"]
    )


MESAJ BAZINDA YAZIM ANALİZİ
-------------------------------------------------------

Mesaj: Şifrenizi hemen doğrulayın.
Tanınmayan kelimeler: []
Tanınmayan kelime oranı: 0.0

Mesaj: Sifrenizi hemen dogrulayin.
Tanınmayan kelimeler: ['Sifrenizi', 'dogrulayin']
Tanınmayan kelime oranı: 0.6667

Mesaj: Hesabınız kapatılacak.
Tanınmayan kelimeler: []
Tanınmayan kelime oranı: 0.0

Mesaj: Hesabnızı kapatılcak.
Tanınmayan kelimeler: ['Hesabnızı', 'kapatılcak']
Tanınmayan kelime oranı: 1.0

Mesaj: Ödülünüzü almak için tıklayın.
Tanınmayan kelimeler: []
Tanınmayan kelime oranı: 0.0

Mesaj: Odulunuzu almak icin tiklayin.
Tanınmayan kelimeler: ['Odulunuzu', 'icin', 'tiklayin']
Tanınmayan kelime oranı: 0.75

Mesaj: Merhaba, yarın toplantımız var.
Tanınmayan kelimeler: []
Tanınmayan kelime oranı: 0.0

Mesaj: Merhabaaa, yarın toplntımız var.
Tanınmayan kelimeler: ['Merhabaaa', 'toplntımız']
Tanınmayan kelime oranı: 0.5


In [24]:

# ADIM 7.8: Türkçe karakter eksikliği kontrolü

from itertools import product

karakter_secenekleri = {
    "c": ["c", "ç"],
    "g": ["g", "ğ"],
    "i": ["i", "ı"],
    "o": ["o", "ö"],
    "s": ["s", "ş"],
    "u": ["u", "ü"]
}

def karakter_eksikligi_kontrol(kelime):

    kelime = turkce_kucult(kelime)

    # Kelime zaten tanınıyorsa öneri üretme
    if kelime_analizi(kelime):
        return []

    secenekler = [
        karakter_secenekleri.get(harf, [harf])
        for harf in kelime
    ]

    # Kombinasyon sayısını hesapla
    kombinasyon_sayisi = 1

    for secenek in secenekler:
        kombinasyon_sayisi *= len(secenek)

    # Çok fazla kombinasyonu engelle
    if kombinasyon_sayisi > 4096:
        return []

    oneriler = []

    for harfler in product(*secenekler):

        aday = "".join(harfler)

        if aday != kelime and kelime_analizi(aday):
            oneriler.append(aday)

    return oneriler[:10]


# Kısa kontrol
print("Fonksiyon başarıyla tanımlandı.")
print(
    "sifrenizi →",
    karakter_eksikligi_kontrol("sifrenizi")
)


Fonksiyon başarıyla tanımlandı.
sifrenizi → ['şifrenizi']


In [25]:

# ADIM 7.9: Mesaj bazında Türkçe karakter eksikliği analizi

import re
import pandas as pd

def mesaj_karakter_analizi(mesaj):

    # URL'leri analizden çıkar
    mesaj = re.sub(
        r"(?:https?://|www\.)\S+",
        " ",
        str(mesaj),
        flags=re.IGNORECASE
    )

    # Mesajı kelimelere ayır
    kelimeler = re.findall(
        r"[a-zA-ZçğıöşüÇĞİÖŞÜ]+",
        mesaj
    )

    toplam = len(kelimeler)
    eksik_kelimeler = []

    for kelime in kelimeler:
        kelime = turkce_kucult(kelime)

        # Önce Zemberek ile kontrol et
        if kelime_analizi(kelime):
            continue

        # Türkçe karakter değişimiyle geçerli
        # bir kelime elde ediliyor mu?
        oneriler = karakter_eksikligi_kontrol(kelime)

        if len(oneriler) > 0:
            eksik_kelimeler.append(kelime)

    oran = (
        len(eksik_kelimeler) / toplam
        if toplam > 0 else 0.0
    )

    return pd.Series({
        "karakter_eksikligi_sayisi": len(eksik_kelimeler),
        "karakter_eksikligi_orani": round(oran, 4),
        "karakter_eksikligi_kelimeleri": eksik_kelimeler
    })


# Veri setindeki mesajları analiz et
sonuclar = df["mesaj"].apply(mesaj_karakter_analizi)

# Yeni sütunları ekle
for sutun in sonuclar.columns:
    df[sutun] = sonuclar[sutun]

display(df[[
    "mesaj",
    "etiket",
    "karakter_eksikligi_sayisi",
    "karakter_eksikligi_orani",
    "karakter_eksikligi_kelimeleri"
]])


,mesaj,etiket,karakter_eksikligi_sayisi,karakter_eksikligi_orani,karakter_eksikligi_kelimeleri
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,0,0.0,[]
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,0,0.0,[]
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0,0.0,[]
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0,0.0,[]


In [26]:

# ADIM 7.10: Türkçe karakter eksikliği testi

test_mesajlari = [
    "Şifrenizi hemen doğrulayın.",
    "Sifrenizi hemen dogrulayin.",
    "Ödülünüzü almak için tıklayın.",
    "Odulunuzu almak icin tiklayin.",
    "Hesabınız kapatılacak.",
    "Hesabnızı kapatılcak.",
    "Merhaba, yarın toplantımız var."
]

for mesaj in test_mesajlari:
    sonuc = mesaj_karakter_analizi(mesaj)

    print("\nMesaj:", mesaj)
    print(
        "Karakter eksikliği sayısı:",
        sonuc["karakter_eksikligi_sayisi"]
    )
    print(
        "Karakter eksikliği oranı:",
        sonuc["karakter_eksikligi_orani"]
    )
    print(
        "Tespit edilen kelimeler:",
        sonuc["karakter_eksikligi_kelimeleri"]
    )



Mesaj: Şifrenizi hemen doğrulayın.
Karakter eksikliği sayısı: 0
Karakter eksikliği oranı: 0.0
Tespit edilen kelimeler: []

Mesaj: Sifrenizi hemen dogrulayin.
Karakter eksikliği sayısı: 2
Karakter eksikliği oranı: 0.6667
Tespit edilen kelimeler: ['sifrenizi', 'dogrulayin']

Mesaj: Ödülünüzü almak için tıklayın.
Karakter eksikliği sayısı: 0
Karakter eksikliği oranı: 0.0
Tespit edilen kelimeler: []

Mesaj: Odulunuzu almak icin tiklayin.
Karakter eksikliği sayısı: 3
Karakter eksikliği oranı: 0.75
Tespit edilen kelimeler: ['odulunuzu', 'icin', 'tiklayin']

Mesaj: Hesabınız kapatılacak.
Karakter eksikliği sayısı: 0
Karakter eksikliği oranı: 0.0
Tespit edilen kelimeler: []

Mesaj: Hesabnızı kapatılcak.
Karakter eksikliği sayısı: 0
Karakter eksikliği oranı: 0.0
Tespit edilen kelimeler: []

Mesaj: Merhaba, yarın toplantımız var.
Karakter eksikliği sayısı: 0
Karakter eksikliği oranı: 0.0
Tespit edilen kelimeler: []


In [27]:

# ADIM 7.11: Yazım sorunlarını sınıflandırma

import re
import pandas as pd

def yazim_hatasi_analizi(mesaj):

    # URL'leri analizden çıkar
    mesaj = re.sub(
        r"(?:https?://|www\.)\S+",
        " ",
        str(mesaj),
        flags=re.IGNORECASE
    )

    kelimeler = re.findall(
        r"[a-zA-ZçğıöşüÇĞİÖŞÜ]+",
        mesaj
    )

    karakter_eksikligi = []
    diger_taninmayan = []

    for kelime in kelimeler:
        temiz = turkce_kucult(kelime)

        if kelime_analizi(temiz):
            continue

        oneriler = karakter_eksikligi_kontrol(temiz)

        if oneriler:
            karakter_eksikligi.append(kelime)
        else:
            diger_taninmayan.append(kelime)

    toplam = len(kelimeler)

    return pd.Series({
        "karakter_eksikligi_adet": len(karakter_eksikligi),
        "diger_taninmayan_adet": len(diger_taninmayan),
        "diger_taninmayan_orani": round(
            len(diger_taninmayan) / toplam, 4
        ) if toplam else 0.0,
        "diger_taninmayan_kelimeler": diger_taninmayan
    })


# Mevcut veri setine uygula
sonuclar = df["mesaj"].apply(yazim_hatasi_analizi)

for sutun in sonuclar.columns:
    df[sutun] = sonuclar[sutun]

# Hatalı örneklerle test et
test_mesajlari = [
    "Şifrenizi hemen doğrulayın.",
    "Sifrenizi hemen dogrulayin.",
    "Merhabaaa, yarın toplntımız var.",
    "Hesabnızı kapatılcak."
]

for mesaj in test_mesajlari:
    sonuc = yazim_hatasi_analizi(mesaj)

    print("\nMesaj:", mesaj)
    print("Karakter eksikliği:", sonuc["karakter_eksikligi_adet"])
    print("Diğer tanınmayan:", sonuc["diger_taninmayan_adet"])
    print("Tanınmayan kelimeler:", sonuc["diger_taninmayan_kelimeler"])



Mesaj: Şifrenizi hemen doğrulayın.
Karakter eksikliği: 0
Diğer tanınmayan: 0
Tanınmayan kelimeler: []

Mesaj: Sifrenizi hemen dogrulayin.
Karakter eksikliği: 2
Diğer tanınmayan: 0
Tanınmayan kelimeler: []

Mesaj: Merhabaaa, yarın toplntımız var.
Karakter eksikliği: 0
Diğer tanınmayan: 2
Tanınmayan kelimeler: ['Merhabaaa', 'toplntımız']

Mesaj: Hesabnızı kapatılcak.
Karakter eksikliği: 0
Diğer tanınmayan: 2
Tanınmayan kelimeler: ['Hesabnızı', 'kapatılcak']


In [28]:

# ADIM 7.12: Dilsel özelliklerin özet tablosu

dilsel_ozellikler = [
    "mesaj",
    "etiket",
    "turkce_karakter_orani",
    "tanimlanmayan_kelime_orani",
    "karakter_eksikligi_orani",
    "karakter_eksikligi_adet",
    "diger_taninmayan_adet",
    "diger_taninmayan_orani"
]

# Eksik sütunları kontrol et
eksik_sutunlar = [
    sutun for sutun in dilsel_ozellikler
    if sutun not in df.columns
]

if eksik_sutunlar:
    print("Eksik sütunlar:", eksik_sutunlar)
else:
    print("SİBERDİL - DİLSEL ÖZELLİK TABLOSU")
    display(df[dilsel_ozellikler])

    print("\nETİKETLERE GÖRE ORTALAMALAR")

    sayisal_ozellikler = dilsel_ozellikler[2:]

    display(
        df.groupby("etiket")[sayisal_ozellikler].mean()
    )


SİBERDİL - DİLSEL ÖZELLİK TABLOSU


,mesaj,etiket,turkce_karakter_orani,tanimlanmayan_kelime_orani,karakter_eksikligi_orani,karakter_eksikligi_adet,diger_taninmayan_adet,diger_taninmayan_orani
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,0.1702,0.0,0.0,0,0,0.0
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,0.2143,0.0,0.0,0,0,0.0
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0.0938,0.0,0.0,0,0,0.0
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0.1143,0.0,0.0,0,0,0.0



ETİKETLERE GÖRE ORTALAMALAR


,turkce_karakter_orani,tanimlanmayan_kelime_orani,karakter_eksikligi_orani,karakter_eksikligi_adet,diger_taninmayan_adet,diger_taninmayan_orani
etiket,,,,,,
0,0.10405,0.0,0.0,0.0,0.0,0.0
1,0.19225,0.0,0.0,0.0,0.0,0.0


In [29]:

# ADIM 8.1: Gelişmiş URL tespiti

import re

url_deseni = re.compile(
    r"(?<![\w@])"
    r"(?:(?:https?://|www\.)?"
    r"(?:[a-zA-Z0-9-]+\.)+"
    r"[a-zA-Z]{2,}"
    r"(?::\d{2,5})?"
    r"(?:/[^\s<>\"']*)?)",
    re.IGNORECASE
)

def url_bul(mesaj):
    mesaj = str(mesaj)

    bulunanlar = url_deseni.findall(mesaj)

    # Tekrarlanan bağlantıları kaldır
    return list(dict.fromkeys(
        url.rstrip(".,;!?)]}")
        for url in bulunanlar
    ))

# Mevcut URL özelliğini güncelle
df["bulunan_url"] = df["mesaj"].apply(url_bul)

df["url_var"] = df["bulunan_url"].apply(
    lambda x: int(len(x) > 0)
)

# Test mesajları
testler = [
    "Hemen https://ornek.com adresine girin.",
    "www.banka.com.tr sitesini ziyaret edin.",
    "Ödülünüz için ornek.com bağlantısına tıklayın.",
    "turkiye.gov.tr adresini ziyaret edin.",
    "turkiye-gov-dogrulama.com adresine girin.",
    "Yarın toplantımız var."
]

print("SİBERDİL - URL TESPİT TESTİ")
print("-" * 50)

for mesaj in testler:
    print("\nMesaj:", mesaj)
    print("Bulunan URL:", url_bul(mesaj))


SİBERDİL - URL TESPİT TESTİ
--------------------------------------------------

Mesaj: Hemen https://ornek.com adresine girin.
Bulunan URL: ['https://ornek.com']

Mesaj: www.banka.com.tr sitesini ziyaret edin.
Bulunan URL: ['www.banka.com.tr']

Mesaj: Ödülünüz için ornek.com bağlantısına tıklayın.
Bulunan URL: ['ornek.com']

Mesaj: turkiye.gov.tr adresini ziyaret edin.
Bulunan URL: ['turkiye.gov.tr']

Mesaj: turkiye-gov-dogrulama.com adresine girin.
Bulunan URL: ['turkiye-gov-dogrulama.com']

Mesaj: Yarın toplantımız var.
Bulunan URL: []


In [30]:

# ADIM 8.2: URL alan adı analizi

!pip -q install tldextract

import tldextract
from urllib.parse import urlsplit

def alan_adi_analizi(url):

    # Protokol yoksa ekle
    if not url.startswith(("http://", "https://")):
        url = "https://" + url

    ayrismis = urlsplit(url)
    host = (ayrismis.hostname or "").lower()

    sonuc = tldextract.extract(host)

    kayitli_alan_adi = (
        sonuc.domain + "." + sonuc.suffix
        if sonuc.suffix else sonuc.domain
    )

    return {
        "host": host,
        "ana_alan_adi": kayitli_alan_adi,
        "alt_alan_adi": sonuc.subdomain,
        "uzanti": sonuc.suffix
    }

# Test
test_url_listesi = [
    "https://turkiye.gov.tr",
    "https://turkiye-gov-dogrulama.com",
    "https://turkiye.gov.tr.sahte-site.com",
    "https://www.google.com",
    "https://ornek.com"
]

for url in test_url_listesi:
    print("\nURL:", url)
    print(alan_adi_analizi(url))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 2.6 MB/s eta 0:00:00

URL: https://turkiye.gov.tr
{'host': 'turkiye.gov.tr', 'ana_alan_adi': 'turkiye.gov.tr', 'alt_alan_adi': '', 'uzanti': 'gov.tr'}

URL: https://turkiye-gov-dogrulama.com
{'host': 'turkiye-gov-dogrulama.com', 'ana_alan_adi': 'turkiye-gov-dogrulama.com', 'alt_alan_adi': '', 'uzanti': 'com'}

URL: https://turkiye.gov.tr.sahte-site.com
{'host': 'turkiye.gov.tr.sahte-site.com', 'ana_alan_adi': 'sahte-site.com', 'alt_alan_adi': 'turkiye.gov.tr', 'uzanti': 'com'}

URL: https://www.google.com
{'host': 'www.google.com', 'ana_alan_adi': 'google.com', 'alt_alan_adi': 'www', 'uzanti': 'com'}

URL: https://ornek.com
{'host': 'ornek.com', 'ana_alan_adi': 'ornek.com', 'alt_alan_adi': '', 'uzanti': 'com'}


In [31]:

# ADIM 8.3: Kurum taklidi analizi

from difflib import SequenceMatcher
import unicodedata

# Örnek olarak doğrulanmış kurum alan adları
guvenilir_alan_adlari = {
    "e-Devlet": "turkiye.gov.tr",
    "Google": "google.com"
}

def alan_adi_normallestir(metin):
    metin = metin.lower()
    metin = unicodedata.normalize("NFKD", metin)
    metin = "".join(
        h for h in metin
        if not unicodedata.combining(h)
    )
    return metin.replace("-", "").replace(".", "")

def kurum_taklidi_analizi(url):
    bilgi = alan_adi_analizi(url)
    ana_alan = bilgi["ana_alan_adi"]
    host = bilgi["host"]

    # Bilinen gerçek alan adıysa taklit işareti verme
    if ana_alan in guvenilir_alan_adlari.values():
        return {
            "taklit_suphesi": 0,
            "benzer_kurum": None,
            "benzerlik": 1.0
        }

    en_yuksek = 0
    benzer_kurum = None

    for kurum, gercek_alan in guvenilir_alan_adlari.items():
        benzerlik = SequenceMatcher(
            None,
            alan_adi_normallestir(ana_alan),
            alan_adi_normallestir(gercek_alan)
        ).ratio()

        if benzerlik > en_yuksek:
            en_yuksek = benzerlik
            benzer_kurum = kurum

    # Yüksek benzerlik veya alan adı taklidi
    taklit = int(
        en_yuksek >= 0.75
        or any(
            gercek_alan in host and ana_alan != gercek_alan
            for gercek_alan in guvenilir_alan_adlari.values()
        )
    )

    return {
        "taklit_suphesi": taklit,
        "benzer_kurum": benzer_kurum,
        "benzerlik": round(en_yuksek, 3)
    }


# Test adresleri
test_url_listesi = [
    "https://turkiye.gov.tr",
    "https://turkiye-gov-dogrulama.com",
    "https://turkiye.gov.tr.sahte-site.com",
    "https://google.com",
    "https://gooogle.com",
    "https://ornek.com"
]

for url in test_url_listesi:
    print("\nURL:", url)
    print(kurum_taklidi_analizi(url))



URL: https://turkiye.gov.tr
{'taklit_suphesi': 0, 'benzer_kurum': None, 'benzerlik': 1.0}

URL: https://turkiye-gov-dogrulama.com
{'taklit_suphesi': 0, 'benzer_kurum': 'e-Devlet', 'benzerlik': 0.647}

URL: https://turkiye.gov.tr.sahte-site.com
{'taklit_suphesi': 1, 'benzer_kurum': 'Google', 'benzerlik': 0.381}

URL: https://google.com
{'taklit_suphesi': 0, 'benzer_kurum': None, 'benzerlik': 1.0}

URL: https://gooogle.com
{'taklit_suphesi': 1, 'benzer_kurum': 'Google', 'benzerlik': 0.947}

URL: https://ornek.com
{'taklit_suphesi': 0, 'benzer_kurum': 'Google', 'benzerlik': 0.588}


In [32]:

# ADIM 8.3.1: Geliştirilmiş kurum taklidi tespiti

from difflib import SequenceMatcher
import re

def kurum_taklidi_analizi(url):
    bilgi = alan_adi_analizi(url)

    ana_alan = bilgi["ana_alan_adi"].lower()
    host = bilgi["host"].lower()

    for kurum, gercek in guvenilir_alan_adlari.items():
        if ana_alan == gercek:
            return {
                "taklit_suphesi": 0,
                "benzer_kurum": kurum,
                "benzerlik": 1.0
            }

    en_iyi_kurum = None
    en_iyi_puan = 0.0
    supheli_kurum = None

    for kurum, gercek in guvenilir_alan_adlari.items():

        # Gerçek alan adı başka bir alan adının
        # alt alanı olarak kullanılıyor mu?
        if host.endswith("." + gercek):
            continue

        if "." + gercek + "." in "." + host + ".":
            supheli_kurum = kurum
            break

        # Kurumun ana ismini karşılaştır
        kurum_adi = gercek.split(".")[0]
        supheli_ad = ana_alan.split(".")[0]

        # tireyle ayrılmış parçaları incele
        parcalar = re.split(r"[-.]", supheli_ad)

        benzerlikler = [
            SequenceMatcher(None, parca, kurum_adi).ratio()
            for parca in parcalar
        ]

        benzerlikler.append(
            SequenceMatcher(None, supheli_ad, kurum_adi).ratio()
        )

        puan = max(benzerlikler)

        if puan > en_iyi_puan:
            en_iyi_puan = puan
            en_iyi_kurum = kurum

        if puan >= 0.85:
            supheli_kurum = kurum

    return {
        "taklit_suphesi": int(supheli_kurum is not None),
        "benzer_kurum": supheli_kurum or en_iyi_kurum,
        "benzerlik": round(en_iyi_puan, 3)
    }


test_url_listesi = [
    "https://turkiye.gov.tr",
    "https://turkiye-gov-dogrulama.com",
    "https://turkiye.gov.tr.sahte-site.com",
    "https://google.com",
    "https://gooogle.com",
    "https://ornek.com"
]

for url in test_url_listesi:
    print("\nURL:", url)
    print(kurum_taklidi_analizi(url))



URL: https://turkiye.gov.tr
{'taklit_suphesi': 0, 'benzer_kurum': 'e-Devlet', 'benzerlik': 1.0}

URL: https://turkiye-gov-dogrulama.com
{'taklit_suphesi': 1, 'benzer_kurum': 'e-Devlet', 'benzerlik': 1.0}

URL: https://turkiye.gov.tr.sahte-site.com
{'taklit_suphesi': 1, 'benzer_kurum': 'e-Devlet', 'benzerlik': 0.0}

URL: https://google.com
{'taklit_suphesi': 0, 'benzer_kurum': 'Google', 'benzerlik': 1.0}

URL: https://gooogle.com
{'taklit_suphesi': 1, 'benzer_kurum': 'Google', 'benzerlik': 0.923}

URL: https://ornek.com
{'taklit_suphesi': 0, 'benzer_kurum': 'Google', 'benzerlik': 0.364}


In [33]:

# ADIM 8.3.2: Kurum taklidi sonuçlarını iyileştirme

onceki_analiz = kurum_taklidi_analizi

def kurum_taklidi_analizi(url):
    sonuc = onceki_analiz(url)
    bilgi = alan_adi_analizi(url)

    ana_alan = bilgi["ana_alan_adi"].lower()
    host = bilgi["host"].lower()

    # Gerçek alan adı başka bir alan adının
    # altında yanıltıcı şekilde kullanılıyorsa
    for kurum, gercek in guvenilir_alan_adlari.items():
        if ana_alan != gercek and (
            host.startswith(gercek + ".")
            or ("." + gercek + ".") in ("." + host + ".")
        ):
            return {
                "taklit_suphesi": 1,
                "benzer_kurum": kurum,
                "benzerlik": 1.0,
                "tespit_nedeni": "Kurum alan adı taklidi"
            }

    if sonuc["taklit_suphesi"] == 1:
        sonuc["tespit_nedeni"] = "Alan adı benzerliği"
    else:
        sonuc["tespit_nedeni"] = "Belirgin taklit göstergesi yok"

    # Taklit göstergesi bulunmadıysa kurum atama
    if sonuc["taklit_suphesi"] == 0:
        if ana_alan not in guvenilir_alan_adlari.values():
            sonuc["benzer_kurum"] = None

    return sonuc


# Test
for url in test_url_listesi:
    print("\n", url)
    print(kurum_taklidi_analizi(url))



 https://turkiye.gov.tr
{'taklit_suphesi': 0, 'benzer_kurum': 'e-Devlet', 'benzerlik': 1.0, 'tespit_nedeni': 'Belirgin taklit göstergesi yok'}

 https://turkiye-gov-dogrulama.com
{'taklit_suphesi': 1, 'benzer_kurum': 'e-Devlet', 'benzerlik': 1.0, 'tespit_nedeni': 'Alan adı benzerliği'}

 https://turkiye.gov.tr.sahte-site.com
{'taklit_suphesi': 1, 'benzer_kurum': 'e-Devlet', 'benzerlik': 1.0, 'tespit_nedeni': 'Kurum alan adı taklidi'}

 https://google.com
{'taklit_suphesi': 0, 'benzer_kurum': 'Google', 'benzerlik': 1.0, 'tespit_nedeni': 'Belirgin taklit göstergesi yok'}

 https://gooogle.com
{'taklit_suphesi': 1, 'benzer_kurum': 'Google', 'benzerlik': 0.923, 'tespit_nedeni': 'Alan adı benzerliği'}

 https://ornek.com
{'taklit_suphesi': 0, 'benzer_kurum': None, 'benzerlik': 0.364, 'tespit_nedeni': 'Belirgin taklit göstergesi yok'}


In [34]:

# ADIM 8.4: URL analizini mesajlarla birleştirme

import pandas as pd

def mesaj_url_analizi(mesaj):

    url_listesi = url_bul(mesaj)

    if not url_listesi:
        return {
            "url_var": 0,
            "url_sayisi": 0,
            "taklit_suphesi": 0,
            "supheli_url": [],
            "url_tespit_nedenleri": []
        }

    supheli_url = []
    nedenler = []

    for url in url_listesi:

        sonuc = kurum_taklidi_analizi(url)

        if sonuc["taklit_suphesi"] == 1:
            supheli_url.append(url)
            nedenler.append(
                sonuc["tespit_nedeni"]
            )

    return {
        "url_var": 1,
        "url_sayisi": len(url_listesi),
        "taklit_suphesi": int(len(supheli_url) > 0),
        "supheli_url": supheli_url,
        "url_tespit_nedenleri": nedenler
    }


# Mevcut veri setine uygula
url_sonuclari = df["mesaj"].apply(
    mesaj_url_analizi
).apply(pd.Series)

for sutun in url_sonuclari.columns:
    df[sutun] = url_sonuclari[sutun]

display(df[[
    "mesaj",
    "etiket",
    "url_var",
    "url_sayisi",
    "taklit_suphesi",
    "supheli_url",
    "url_tespit_nedenleri"
]])


,mesaj,etiket,url_var,url_sayisi,taklit_suphesi,supheli_url,url_tespit_nedenleri
0,"ACİL! Hesabınız kapatılacak, hemen bağlantıya ...",1,0,0,0,[],[]
1,"Tebrikler ödül kazandınız, ödülünüzü almak içi...",1,0,0,0,[],[]
2,"Merhaba, yarın saat 10'da toplantımız var.",0,0,0,0,[],[]
3,Ödev dosyasını sana e-posta ile gönderdim.,0,0,0,0,[],[]


In [35]:

# ADIM 8.4.1: Mesajların tamamını görüntüleme

for i, satir in df.iterrows():
    print(f"\nMesaj {i + 1}")
    print("İçerik:", satir["mesaj"])
    print("Etiket:", satir["etiket"])
    print("Bulunan URL:", url_bul(satir["mesaj"]))
    print("-" * 60)



Mesaj 1
İçerik: ACİL! Hesabınız kapatılacak, hemen bağlantıya tıklayın!
Etiket: 1
Bulunan URL: []
------------------------------------------------------------

Mesaj 2
İçerik: Tebrikler ödül kazandınız, ödülünüzü almak için hemen işlem yapın!
Etiket: 1
Bulunan URL: []
------------------------------------------------------------

Mesaj 3
İçerik: Merhaba, yarın saat 10'da toplantımız var.
Etiket: 0
Bulunan URL: []
------------------------------------------------------------

Mesaj 4
İçerik: Ödev dosyasını sana e-posta ile gönderdim.
Etiket: 0
Bulunan URL: []
------------------------------------------------------------


In [36]:

# ADIM 8.5: URL içeren ve içermeyen mesajların testi

import pandas as pd

test_verileri = [
    {
        "mesaj": "Hesabınız kapatılacak! https://turkiye-gov-dogrulama.com adresine girin.",
        "etiket": 1
    },
    {
        "mesaj": "Hesabınızı doğrulamak için https://turkiye.gov.tr.sahte-site.com adresini ziyaret edin.",
        "etiket": 1
    },
    {
        "mesaj": "Google hesabınız için https://gooogle.com adresine giriş yapın.",
        "etiket": 1
    },
    {
        "mesaj": "e-Devlet hizmetlerine https://turkiye.gov.tr adresinden ulaşabilirsiniz.",
        "etiket": 0
    },
    {
        "mesaj": "Yarın saat 10'da toplantımız var.",
        "etiket": 0
    },
    {
        "mesaj": "ACİL! Hesabınız kapatılacak, hemen şifrenizi gönderin!",
        "etiket": 1
    }
]

df_test = pd.DataFrame(test_verileri)

url_sonuclari = df_test["mesaj"].apply(
    mesaj_url_analizi
).apply(pd.Series)

df_test = pd.concat(
    [df_test, url_sonuclari],
    axis=1
)

display(df_test[[
    "mesaj",
    "etiket",
    "url_var",
    "url_sayisi",
    "taklit_suphesi",
    "supheli_url"
]])


,mesaj,etiket,url_var,url_sayisi,taklit_suphesi,supheli_url
0,Hesabınız kapatılacak! https://turkiye-gov-dog...,1,1,1,1,[https://turkiye-gov-dogrulama.com]
1,Hesabınızı doğrulamak için https://turkiye.gov...,1,1,1,1,[https://turkiye.gov.tr.sahte-site.com]
2,Google hesabınız için https://gooogle.com adre...,1,1,1,1,[https://gooogle.com]
3,e-Devlet hizmetlerine https://turkiye.gov.tr a...,0,1,1,0,[]
4,Yarın saat 10'da toplantımız var.,0,0,0,0,[]
5,"ACİL! Hesabınız kapatılacak, hemen şifrenizi g...",1,0,0,0,[]


In [37]:

# ADIM 8.6: SiberDil birleşik özellik analizi

import pandas as pd

def siberdil_analizi(mesaj):

    # 1. URL analizi
    url_sonuc = mesaj_url_analizi(mesaj)

    # 2. URL'leri dil analizinden çıkar
    import re
    temiz_mesaj = re.sub(
        r"(?:https?://|www\.)\S+",
        " ",
        str(mesaj),
        flags=re.IGNORECASE
    )

    # 3. Dilsel analizler
    yazim = yazim_hatasi_analizi(temiz_mesaj)

    sonuc = {
        "mesaj": mesaj,
        "url_var": url_sonuc["url_var"],
        "url_sayisi": url_sonuc["url_sayisi"],
        "taklit_suphesi": url_sonuc["taklit_suphesi"],
        "karakter_eksikligi": int(
            yazim["karakter_eksikligi_adet"]
        ),
        "diger_taninmayan": int(
            yazim["diger_taninmayan_adet"]
        ),
        "diger_taninmayan_orani": float(
            yazim["diger_taninmayan_orani"]
        )
    }

    return sonuc


# Altı test mesajına uygula
birlesik_sonuclar = pd.DataFrame([
    siberdil_analizi(mesaj)
    for mesaj in df_test["mesaj"]
])

birlesik_sonuclar["etiket"] = df_test["etiket"].values

display(birlesik_sonuclar)


,mesaj,url_var,url_sayisi,taklit_suphesi,karakter_eksikligi,diger_taninmayan,diger_taninmayan_orani,etiket
0,Hesabınız kapatılacak! https://turkiye-gov-dog...,1,1,1,0,0,0.0,1
1,Hesabınızı doğrulamak için https://turkiye.gov...,1,1,1,0,0,0.0,1
2,Google hesabınız için https://gooogle.com adre...,1,1,1,0,0,0.0,1
3,e-Devlet hizmetlerine https://turkiye.gov.tr a...,1,1,0,0,0,0.0,0
4,Yarın saat 10'da toplantımız var.,0,0,0,0,0,0.0,0
5,"ACİL! Hesabınız kapatılacak, hemen şifrenizi g...",0,0,0,0,0,0.0,1


In [38]:

# ADIM 8.7: Mesaj içeriğindeki risk göstergeleri

import re
import pandas as pd

def icerik_risk_analizi(mesaj):

    # URL'leri içerik analizinden çıkar
    url_listesi = url_bul(mesaj)
    temiz_mesaj = str(mesaj)

    for url in url_listesi:
        temiz_mesaj = temiz_mesaj.replace(url, " ")

    metin = turkce_kucult(temiz_mesaj)

    # Aciliyet ifadeleri
    aciliyet_kaliplari = [
        r"\bacil\b",
        r"\bhemen\b",
        r"\bderhal\b",
        r"\bvakit kaybetmeden\b",
        r"\bson şans\b",
        r"\b\d+\s*saat içinde\b"
    ]

    # Hesap tehdidi ifadeleri
    tehdit_kaliplari = [
        r"hesabınız.{0,40}kapatıl",
        r"hesabınız.{0,40}askıya alın",
        r"hesabınız.{0,40}engellen",
        r"erişiminiz.{0,40}durdurul"
    ]

    # Ödül vaadi ifadeleri
    odul_kaliplari = [
        r"ödül kazand",
        r"ödülünüzü al",
        r"hediye kazand",
        r"çekiliş.{0,30}kazand"
    ]

    # Hassas bilgi talebi
    bilgi_kaliplari = [
        r"şifrenizi.{0,30}(gönder|paylaş|gir|yaz)",
        r"doğrulama kodunuzu.{0,30}(gönder|paylaş|gir|yaz)",
        r"kimlik numaranızı.{0,30}(gönder|paylaş|gir|yaz)",
        r"kart bilgilerinizi.{0,30}(gönder|paylaş|gir|yaz)"
    ]

    def kontrol(kaliplar):
        return int(any(
            re.search(kalip, metin)
            for kalip in kaliplar
        ))

    return {
        "aciliyet": kontrol(aciliyet_kaliplari),
        "hesap_tehdidi": kontrol(tehdit_kaliplari),
        "odul_vaadi": kontrol(odul_kaliplari),
        "bilgi_talebi": kontrol(bilgi_kaliplari)
    }


# Önceki altı test mesajına uygula
icerik_sonuclari = pd.DataFrame([
    icerik_risk_analizi(mesaj)
    for mesaj in df_test["mesaj"]
])

display(pd.concat([
    df_test[["mesaj", "etiket"]].reset_index(drop=True),
    icerik_sonuclari
], axis=1))


,mesaj,etiket,aciliyet,hesap_tehdidi,odul_vaadi,bilgi_talebi
0,Hesabınız kapatılacak! https://turkiye-gov-dog...,1,0,1,0,0
1,Hesabınızı doğrulamak için https://turkiye.gov...,1,0,0,0,0
2,Google hesabınız için https://gooogle.com adre...,1,0,0,0,0
3,e-Devlet hizmetlerine https://turkiye.gov.tr a...,0,0,0,0,0
4,Yarın saat 10'da toplantımız var.,0,0,0,0,0
5,"ACİL! Hesabınız kapatılacak, hemen şifrenizi g...",1,1,1,0,1


In [39]:

# ADIM 8.8: SiberDil birleşik analiz sistemi

import pandas as pd

def siberdil_tam_analiz(mesaj):

    # 1. URL ve yazım analizi
    temel = siberdil_analizi(mesaj)

    # 2. İçerik analizi
    icerik = icerik_risk_analizi(mesaj)

    # 3. Bütün özellikleri birleştir
    sonuc = {**temel, **icerik}

    return sonuc


# Altı test mesajına uygula
tam_sonuclar = pd.DataFrame([
    siberdil_tam_analiz(mesaj)
    for mesaj in df_test["mesaj"]
])

# Gerçek etiketleri yalnızca karşılaştırma için ekle
tam_sonuclar["etiket"] = df_test["etiket"].values

# Sütun sıralaması
sutunlar = [
    "mesaj",
    "etiket",
    "url_var",
    "url_sayisi",
    "taklit_suphesi",
    "aciliyet",
    "hesap_tehdidi",
    "odul_vaadi",
    "bilgi_talebi",
    "karakter_eksikligi",
    "diger_taninmayan",
    "diger_taninmayan_orani"
]

display(tam_sonuclar[sutunlar])


,mesaj,etiket,url_var,url_sayisi,taklit_suphesi,aciliyet,hesap_tehdidi,odul_vaadi,bilgi_talebi,karakter_eksikligi,diger_taninmayan,diger_taninmayan_orani
0,Hesabınız kapatılacak! https://turkiye-gov-dog...,1,1,1,1,0,1,0,0,0,0,0.0
1,Hesabınızı doğrulamak için https://turkiye.gov...,1,1,1,1,0,0,0,0,0,0,0.0
2,Google hesabınız için https://gooogle.com adre...,1,1,1,1,0,0,0,0,0,0,0.0
3,e-Devlet hizmetlerine https://turkiye.gov.tr a...,0,1,1,0,0,0,0,0,0,0,0.0
4,Yarın saat 10'da toplantımız var.,0,0,0,0,0,0,0,0,0,0,0.0
5,"ACİL! Hesabınız kapatılacak, hemen şifrenizi g...",1,0,0,0,1,1,0,1,0,0,0.0


In [40]:

# ADIM 8.9: SiberDil deneysel risk puanlama

def risk_puani_hesapla(satir):

    puan = 0

    puan += int(satir["taklit_suphesi"]) * 45
    puan += int(satir["bilgi_talebi"]) * 30
    puan += int(satir["hesap_tehdidi"]) * 15
    puan += int(satir["aciliyet"]) * 10
    puan += int(satir["odul_vaadi"]) * 10

    if satir["karakter_eksikligi"] > 0:
        puan += 5

    if satir["diger_taninmayan"] > 0:
        puan += 5

    return min(puan, 100)


def risk_seviyesi_belirle(puan):

    if puan >= 60:
        return "Yüksek"

    elif puan >= 30:
        return "Orta"

    return "Düşük"


tam_sonuclar["risk_puani"] = tam_sonuclar.apply(
    risk_puani_hesapla,
    axis=1
)

tam_sonuclar["risk_seviyesi"] = (
    tam_sonuclar["risk_puani"].apply(
        risk_seviyesi_belirle
    )
)

display(tam_sonuclar[[
    "mesaj",
    "etiket",
    "risk_puani",
    "risk_seviyesi"
]])


,mesaj,etiket,risk_puani,risk_seviyesi
0,Hesabınız kapatılacak! https://turkiye-gov-dog...,1,60,Yüksek
1,Hesabınızı doğrulamak için https://turkiye.gov...,1,45,Orta
2,Google hesabınız için https://gooogle.com adre...,1,45,Orta
3,e-Devlet hizmetlerine https://turkiye.gov.tr a...,0,0,Düşük
4,Yarın saat 10'da toplantımız var.,0,0,Düşük
5,"ACİL! Hesabınız kapatılacak, hemen şifrenizi g...",1,55,Orta


In [41]:

# ADIM 8.10: Risk puanı gerekçelerini oluşturma

def risk_gerekceleri(satir):

    gerekceler = []

    if satir["taklit_suphesi"] == 1:
        gerekceler.append(
            "Bilinen bir kurumun alan adına benzeyen bağlantı"
        )

    if satir["bilgi_talebi"] == 1:
        gerekceler.append(
            "Hassas bilgi veya şifre talebi"
        )

    if satir["hesap_tehdidi"] == 1:
        gerekceler.append(
            "Hesabın kapatılması veya engellenmesi tehdidi"
        )

    if satir["aciliyet"] == 1:
        gerekceler.append(
            "Acil işlem yapmaya yönlendirme"
        )

    if satir["odul_vaadi"] == 1:
        gerekceler.append(
            "Ödül veya hediye vaadi"
        )

    if satir["karakter_eksikligi"] > 0:
        gerekceler.append(
            "Olası Türkçe karakter eksiklikleri"
        )

    if satir["diger_taninmayan"] > 0:
        gerekceler.append(
            "Zemberek tarafından tanınmayan kelimeler"
        )

    if not gerekceler:
        gerekceler.append(
            "Mevcut kurallarla belirgin risk göstergesi bulunamadı"
        )

    return gerekceler


tam_sonuclar["risk_gerekceleri"] = tam_sonuclar.apply(
    risk_gerekceleri,
    axis=1
)

for _, satir in tam_sonuclar.iterrows():

    print("\nMESAJ:", satir["mesaj"])
    print("RİSK PUANI:", satir["risk_puani"])
    print("RİSK SEVİYESİ:", satir["risk_seviyesi"])

    print("GEREKÇELER:")

    for gerekce in satir["risk_gerekceleri"]:
        print("-", gerekce)

    print("-" * 60)



MESAJ: Hesabınız kapatılacak! https://turkiye-gov-dogrulama.com adresine girin.
RİSK PUANI: 60
RİSK SEVİYESİ: Yüksek
GEREKÇELER:
- Bilinen bir kurumun alan adına benzeyen bağlantı
- Hesabın kapatılması veya engellenmesi tehdidi
------------------------------------------------------------

MESAJ: Hesabınızı doğrulamak için https://turkiye.gov.tr.sahte-site.com adresini ziyaret edin.
RİSK PUANI: 45
RİSK SEVİYESİ: Orta
GEREKÇELER:
- Bilinen bir kurumun alan adına benzeyen bağlantı
------------------------------------------------------------

MESAJ: Google hesabınız için https://gooogle.com adresine giriş yapın.
RİSK PUANI: 45
RİSK SEVİYESİ: Orta
GEREKÇELER:
- Bilinen bir kurumun alan adına benzeyen bağlantı
------------------------------------------------------------

MESAJ: e-Devlet hizmetlerine https://turkiye.gov.tr adresinden ulaşabilirsiniz.
RİSK PUANI: 0
RİSK SEVİYESİ: Düşük
GEREKÇELER:
- Mevcut kurallarla belirgin risk göstergesi bulunamadı
----------------------------------------

In [42]:

# ADIM 8.11: Birleşik risk göstergeleri

def risk_puani_hesapla_v2(satir):

    puan = 0

    taklit = int(satir["taklit_suphesi"])
    bilgi = int(satir["bilgi_talebi"])
    tehdit = int(satir["hesap_tehdidi"])
    aciliyet = int(satir["aciliyet"])
    odul = int(satir["odul_vaadi"])

    # Temel göstergeler
    puan += taklit * 45
    puan += bilgi * 30
    puan += tehdit * 15
    puan += aciliyet * 10
    puan += odul * 10

    # Dilsel göstergeler
    if satir["karakter_eksikligi"] > 0:
        puan += 5

    if satir["diger_taninmayan"] > 0:
        puan += 5

    # Birlikte görülen göstergeler
    if bilgi and tehdit:
        puan += 20

    if taklit and tehdit:
        puan += 15

    if taklit and bilgi:
        puan += 20

    if odul and bilgi:
        puan += 10

    return min(puan, 100)


# Yeni puanları hesapla
tam_sonuclar["risk_puani_v2"] = tam_sonuclar.apply(
    risk_puani_hesapla_v2,
    axis=1
)

tam_sonuclar["risk_seviyesi_v2"] = (
    tam_sonuclar["risk_puani_v2"].apply(
        risk_seviyesi_belirle
    )
)

# Eski ve yeni puanları karşılaştır
display(tam_sonuclar[[
    "mesaj",
    "etiket",
    "risk_puani",
    "risk_seviyesi",
    "risk_puani_v2",
    "risk_seviyesi_v2"
]])


,mesaj,etiket,risk_puani,risk_seviyesi,risk_puani_v2,risk_seviyesi_v2
0,Hesabınız kapatılacak! https://turkiye-gov-dog...,1,60,Yüksek,75,Yüksek
1,Hesabınızı doğrulamak için https://turkiye.gov...,1,45,Orta,45,Orta
2,Google hesabınız için https://gooogle.com adre...,1,45,Orta,45,Orta
3,e-Devlet hizmetlerine https://turkiye.gov.tr a...,0,0,Düşük,0,Düşük
4,Yarın saat 10'da toplantımız var.,0,0,Düşük,0,Düşük
5,"ACİL! Hesabınız kapatılacak, hemen şifrenizi g...",1,55,Orta,75,Yüksek


In [43]:

# ADIM 8.12: SiberDil risk sistemi kontrol testi

import pandas as pd

yeni_testler = [
    ("Hemen sınıfa gel, ders başlıyor.", 0),
    ("Tebrikler, yarışmada ödül kazandınız!", 0),
    ("Hesabınız kapatılacak, şifrenizi gönderin!", 1),
    ("Şifrenizi kimseyle paylaşmayın.", 0),
    ("https://turkiye.gov.tr adresinden işlem yapabilirsiniz.", 0),
    ("https://turkiye-gov-dogrulama.com adresine girin.", 1),
    ("Sifrenizi hemen gonderin!", 1),
    ("Yarın saat 14.00'te toplantımız var.", 0),
    ("Google hesabınız askıya alınacak, hemen giriş yapın!", 1),
    ("Ödülünüzü almak için kimlik numaranızı paylaşın.", 1)
]

test_sonuclari = []

for mesaj, gercek_etiket in yeni_testler:

    analiz = siberdil_tam_analiz(mesaj)

    puan = risk_puani_hesapla_v2(analiz)

    test_sonuclari.append({
        "mesaj": mesaj,
        "gercek_etiket": gercek_etiket,
        "risk_puani": puan,
        "risk_seviyesi": risk_seviyesi_belirle(puan)
    })

df_kontrol = pd.DataFrame(test_sonuclari)

display(df_kontrol)


,mesaj,gercek_etiket,risk_puani,risk_seviyesi
0,"Hemen sınıfa gel, ders başlıyor.",0,10,Düşük
1,"Tebrikler, yarışmada ödül kazandınız!",0,10,Düşük
2,"Hesabınız kapatılacak, şifrenizi gönderin!",1,65,Yüksek
3,Şifrenizi kimseyle paylaşmayın.,0,30,Orta
4,https://turkiye.gov.tr adresinden işlem yapabi...,0,0,Düşük
5,https://turkiye-gov-dogrulama.com adresine girin.,1,45,Orta
6,Sifrenizi hemen gonderin!,1,15,Düşük
7,Yarın saat 14.00'te toplantımız var.,0,0,Düşük
8,"Google hesabınız askıya alınacak, hemen giriş ...",1,25,Düşük
9,Ödülünüzü almak için kimlik numaranızı paylaşın.,1,50,Orta


In [44]:

# ADIM 8.13: Geliştirilmiş hassas bilgi talebi analizi

import re
import unicodedata

def turkce_ascii(metin):
    metin = turkce_kucult(str(metin))
    metin = metin.replace("ı", "i")
    return "".join(
        h for h in unicodedata.normalize("NFKD", metin)
        if not unicodedata.combining(h)
    )

def bilgi_talebi_v2(mesaj):
    metin = turkce_ascii(mesaj)

    # Hassas bilgi ifadeleri
    hassas_bilgiler = [
        r"sifreniz\w*",
        r"dogrulama\s+kodun\w*",
        r"kimlik\s+numaraniz\w*",
        r"kart\s+bilgileriniz\w*"
    ]

    # Talep bildiren fiiller
    talep_fiilleri = [
        r"gonder\w*",
        r"paylas\w*",
        r"gir\w*",
        r"yaz\w*"
    ]

    # Açık olumsuz emirleri hariç tut
    olumsuz_emir = re.search(
        r"\b(?:paylasma|gonderme|girme|yazma)"
        r"(?:yin|yiniz|yın|yınız|n|nız|niz)?\b",
        metin
    )

    if olumsuz_emir:
        return 0

    hassas_var = any(
        re.search(k, metin) for k in hassas_bilgiler
    )

    talep_var = any(
        re.search(k, metin) for k in talep_fiilleri
    )

    return int(hassas_var and talep_var)


# Kontrol testleri
testler = [
    "Şifrenizi kimseyle paylaşmayın.",
    "Sifrenizi hemen gonderin!",
    "Şifrenizi gönderin!",
    "Ödülünüzü almak için kimlik numaranızı paylaşın.",
    "Yarın toplantımız var."
]

for mesaj in testler:
    print(mesaj, "→", bilgi_talebi_v2(mesaj))


Şifrenizi kimseyle paylaşmayın. → 0
Sifrenizi hemen gonderin! → 1
Şifrenizi gönderin! → 1
Ödülünüzü almak için kimlik numaranızı paylaşın. → 1
Yarın toplantımız var. → 0


In [45]:

# ADIM 8.14: Geliştirilmiş hesap tehdidi analizi

import re

def hesap_tehdidi_v2(mesaj):

    metin = turkce_ascii(mesaj)

    # Hesap ve erişim ifadeleri
    hesap_kaliplari = [
        r"\bhesab\w*",
        r"\bhesap\w*",
        r"\berisim\w*",
        r"\boturum\w*"
    ]

    # Hesap kapatma veya kısıtlama ifadeleri
    tehdit_kaliplari = [
        r"kapatil\w*",
        r"askiya\s+alin\w*",
        r"engellen\w*",
        r"bloke\s+edil\w*",
        r"durdurul\w*",
        r"kilitlen\w*"
    ]

    hesap_eslesmeleri = [
        esleme
        for kalip in hesap_kaliplari
        for esleme in re.finditer(kalip, metin)
    ]

    tehdit_eslesmeleri = [
        esleme
        for kalip in tehdit_kaliplari
        for esleme in re.finditer(kalip, metin)
    ]

    # Hesap ve tehdit ifadesi birbirine yakın mı?
    for hesap in hesap_eslesmeleri:
        for tehdit in tehdit_eslesmeleri:
            mesafe = max(
                hesap.start() - tehdit.end(),
                tehdit.start() - hesap.end(),
                0
            )

            if mesafe <= 60:
                return 1

    return 0


testler = [
    "Google hesabınız askıya alınacak, hemen giriş yapın!",
    "Hesabınız kapatılacak!",
    "Hesabiniz bloke edilecek.",
    "Erişiminiz durdurulacak.",
    "Yarın toplantımız var.",
    "Hesabınız başarıyla oluşturuldu."
]

for mesaj in testler:
    print(mesaj, "→", hesap_tehdidi_v2(mesaj))


Google hesabınız askıya alınacak, hemen giriş yapın! → 1
Hesabınız kapatılacak! → 1
Hesabiniz bloke edilecek. → 1
Erişiminiz durdurulacak. → 1
Yarın toplantımız var. → 0
Hesabınız başarıyla oluşturuldu. → 0


In [46]:

# ADIM 8.15: SiberDil V2 analiz sistemi

import pandas as pd

def siberdil_analiz_v2(mesaj):

    # Önceki birleşik analiz
    sonuc = siberdil_tam_analiz(mesaj)

    # URL'leri metinden çıkar
    temiz_mesaj = str(mesaj)

    for url in url_bul(temiz_mesaj):
        temiz_mesaj = temiz_mesaj.replace(url, " ")

    # Geliştirilmiş dilsel kontroller
    sonuc["bilgi_talebi"] = bilgi_talebi_v2(temiz_mesaj)
    sonuc["hesap_tehdidi"] = hesap_tehdidi_v2(temiz_mesaj)

    # Yeni risk puanı
    sonuc["risk_puani"] = risk_puani_hesapla_v2(sonuc)

    sonuc["risk_seviyesi"] = risk_seviyesi_belirle(
        sonuc["risk_puani"]
    )

    return sonuc


# Önceki 10 mesajlık kontrol veri setini yeniden analiz et
v2_sonuclar = []

for _, satir in df_kontrol.iterrows():

    sonuc = siberdil_analiz_v2(satir["mesaj"])

    v2_sonuclar.append({
        "mesaj": satir["mesaj"],
        "gercek_etiket": satir["gercek_etiket"],
        "eski_puan": satir["risk_puani"],
        "yeni_puan": sonuc["risk_puani"],
        "yeni_seviye": sonuc["risk_seviyesi"],
        "bilgi_talebi": sonuc["bilgi_talebi"],
        "hesap_tehdidi": sonuc["hesap_tehdidi"]
    })

df_v2 = pd.DataFrame(v2_sonuclar)

display(df_v2)


,mesaj,gercek_etiket,eski_puan,yeni_puan,yeni_seviye,bilgi_talebi,hesap_tehdidi
0,"Hemen sınıfa gel, ders başlıyor.",0,10,10,Düşük,0,0
1,"Tebrikler, yarışmada ödül kazandınız!",0,10,10,Düşük,0,0
2,"Hesabınız kapatılacak, şifrenizi gönderin!",1,65,65,Yüksek,1,1
3,Şifrenizi kimseyle paylaşmayın.,0,30,0,Düşük,0,0
4,https://turkiye.gov.tr adresinden işlem yapabi...,0,0,0,Düşük,0,0
5,https://turkiye-gov-dogrulama.com adresine girin.,1,45,45,Orta,0,0
6,Sifrenizi hemen gonderin!,1,15,45,Orta,1,0
7,Yarın saat 14.00'te toplantımız var.,0,0,0,Düşük,0,0
8,"Google hesabınız askıya alınacak, hemen giriş ...",1,25,25,Düşük,0,1
9,Ödülünüzü almak için kimlik numaranızı paylaşın.,1,50,50,Orta,1,0


In [47]:

# ADIM 8.16: SiberDil V3 risk puanlama

def risk_puani_hesapla_v3(satir):

    # Önceki V2 puanını al
    puan = risk_puani_hesapla_v2(satir)

    bilgi = int(satir["bilgi_talebi"])
    tehdit = int(satir["hesap_tehdidi"])
    aciliyet = int(satir["aciliyet"])

    # Yeni birleşik göstergeler
    if tehdit and aciliyet:
        puan += 20

    if bilgi and aciliyet:
        puan += 15

    return min(puan, 100)


# Mevcut 10 mesajı yeniden analiz et
v3_sonuclar = []

for _, satir in df_kontrol.iterrows():

    analiz = siberdil_analiz_v2(satir["mesaj"])

    yeni_puan = risk_puani_hesapla_v3(analiz)

    v3_sonuclar.append({
        "mesaj": satir["mesaj"],
        "gercek_etiket": satir["gercek_etiket"],
        "v2_puan": analiz["risk_puani"],
        "v3_puan": yeni_puan,
        "v3_seviye": risk_seviyesi_belirle(yeni_puan)
    })

df_v3 = pd.DataFrame(v3_sonuclar)

display(df_v3)


,mesaj,gercek_etiket,v2_puan,v3_puan,v3_seviye
0,"Hemen sınıfa gel, ders başlıyor.",0,10,10,Düşük
1,"Tebrikler, yarışmada ödül kazandınız!",0,10,10,Düşük
2,"Hesabınız kapatılacak, şifrenizi gönderin!",1,65,65,Yüksek
3,Şifrenizi kimseyle paylaşmayın.,0,0,0,Düşük
4,https://turkiye.gov.tr adresinden işlem yapabi...,0,0,0,Düşük
5,https://turkiye-gov-dogrulama.com adresine girin.,1,45,45,Orta
6,Sifrenizi hemen gonderin!,1,45,60,Yüksek
7,Yarın saat 14.00'te toplantımız var.,0,0,0,Düşük
8,"Google hesabınız askıya alınacak, hemen giriş ...",1,25,45,Orta
9,Ödülünüzü almak için kimlik numaranızı paylaşın.,1,50,50,Orta


In [48]:

# ADIM 8.17: SiberDil V3 performans değerlendirmesi

import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Gerçek etiketler
y_true = df_v3["gercek_etiket"]

# Deneysel karar eşiği
ESIK = 30

# Algoritmanın tahminleri
y_pred = (df_v3["v3_puan"] >= ESIK).astype(int)

# Performans ölçümleri
print("SİBERDİL V3 PERFORMANS RAPORU")
print("-" * 45)

print("Accuracy:",
      round(accuracy_score(y_true, y_pred), 3))

print("Precision:",
      round(precision_score(y_true, y_pred, zero_division=0), 3))

print("Recall:",
      round(recall_score(y_true, y_pred, zero_division=0), 3))

print("F1-score:",
      round(f1_score(y_true, y_pred, zero_division=0), 3))

print("\nKARMAŞIKLIK MATRİSİ")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nDETAYLI RAPOR")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Normal", "Oltalama"],
    zero_division=0
))

# Hatalı sınıflandırılan mesajlar
kontrol = df_v3.copy()
kontrol["tahmin"] = y_pred

hatalar = kontrol[
    kontrol["gercek_etiket"] != kontrol["tahmin"]
]

print("\nHATALI SINIFLANDIRILAN MESAJLAR")
display(hatalar)


SİBERDİL V3 PERFORMANS RAPORU
---------------------------------------------
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0

KARMAŞIKLIK MATRİSİ
[[5 0]
 [0 5]]

DETAYLI RAPOR
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00         5
    Oltalama       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10


HATALI SINIFLANDIRILAN MESAJLAR


,mesaj,gercek_etiket,v2_puan,v3_puan,v3_seviye,tahmin


In [49]:

# ADIM 8.18: Yeni mesajlarla dayanıklılık testi

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

zor_testler = [
    ("Bankanızdan gelen mesajları kimseyle paylaşmayın.", 0),
    ("Hesabınız güvenlik nedeniyle geçici olarak kapatılmıştır.", 0),
    ("Ödül töreni yarın saat 14.00'te başlayacak.", 0),
    ("Şifrenizi değiştirmek için hesap ayarlarına gidin.", 0),
    ("Lütfen doğrulama kodunuzu bizimle paylaşınız.", 1),
    ("Hesabiniz bloke olacak, bilgilerinizi guncelleyin.", 1),
    ("Kimlik bilgilerinizi hemen bu numaraya iletin.", 1),
    ("https://google.com üzerinden arama yapabilirsiniz.", 0),
    ("https://gooogle.com adresinden hesabınızı doğrulayın.", 1),
    ("Sayın müşterimiz, güvenlik kontrolü için kart bilgilerinizi giriniz.", 1),
    ("Hemen gel, sana bir sürprizim var!", 0),
    ("Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.", 1),
    ("Hesabınızla ilgili bilgilendirme mesajıdır.", 0),
    ("Sifrenizi asla kimseye gondermeyin.", 0),
    ("Banka hesabınızın kapatılmaması için hemen işlem yapın.", 1)
]

kayitlar = []

for mesaj, etiket in zor_testler:
    analiz = siberdil_analiz_v2(mesaj)
    puan = risk_puani_hesapla_v3(analiz)
    tahmin = int(puan >= 30)

    kayitlar.append({
        "mesaj": mesaj,
        "gercek_etiket": etiket,
        "risk_puani": puan,
        "tahmin": tahmin,
        "dogru_mu": etiket == tahmin
    })

df_zor = pd.DataFrame(kayitlar)

display(df_zor)

print("\nKARMAŞIKLIK MATRİSİ")
print(confusion_matrix(
    df_zor["gercek_etiket"],
    df_zor["tahmin"],
    labels=[0, 1]
))

print("\nPERFORMANS RAPORU")
print(classification_report(
    df_zor["gercek_etiket"],
    df_zor["tahmin"],
    zero_division=0
))


,mesaj,gercek_etiket,risk_puani,tahmin,dogru_mu
0,Bankanızdan gelen mesajları kimseyle paylaşmayın.,0,0,0,True
1,Hesabınız güvenlik nedeniyle geçici olarak kap...,0,15,0,True
2,Ödül töreni yarın saat 14.00'te başlayacak.,0,0,0,True
3,Şifrenizi değiştirmek için hesap ayarlarına gi...,0,0,0,True
4,Lütfen doğrulama kodunuzu bizimle paylaşınız.,1,30,1,True
5,"Hesabiniz bloke olacak, bilgilerinizi guncelle...",1,5,0,False
6,Kimlik bilgilerinizi hemen bu numaraya iletin.,1,10,0,False
7,https://google.com üzerinden arama yapabilirsi...,0,0,0,True
8,https://gooogle.com adresinden hesabınızı doğr...,1,45,1,True
9,"Sayın müşterimiz, güvenlik kontrolü için kart ...",1,30,1,True



KARMAŞIKLIK MATRİSİ
[[8 0]
 [3 4]]

PERFORMANS RAPORU
              precision    recall  f1-score   support

           0       0.73      1.00      0.84         8
           1       1.00      0.57      0.73         7

    accuracy                           0.80        15
   macro avg       0.86      0.79      0.78        15
weighted avg       0.85      0.80      0.79        15



In [50]:

# ADIM 8.19: Kaçırılan mesajların özellik analizi

import pandas as pd

kacirilanlar = df_zor[
    (df_zor["gercek_etiket"] == 1) &
    (df_zor["tahmin"] == 0)
]

kayitlar = []

for mesaj in kacirilanlar["mesaj"]:

    analiz = siberdil_analiz_v2(mesaj)

    kayitlar.append({
        "mesaj": mesaj,
        "aciliyet": analiz["aciliyet"],
        "hesap_tehdidi": analiz["hesap_tehdidi"],
        "bilgi_talebi": analiz["bilgi_talebi"],
        "odul_vaadi": analiz["odul_vaadi"],
        "karakter_eksikligi": analiz["karakter_eksikligi"],
        "diger_taninmayan": analiz["diger_taninmayan"]
    })

df_hata_analizi = pd.DataFrame(kayitlar)

pd.set_option("display.max_colwidth", 150)

display(df_hata_analizi)


,mesaj,aciliyet,hesap_tehdidi,bilgi_talebi,odul_vaadi,karakter_eksikligi,diger_taninmayan
0,"Hesabiniz bloke olacak, bilgilerinizi guncelleyin.",0,0,0,0,1,0
1,Kimlik bilgilerinizi hemen bu numaraya iletin.,1,0,0,0,0,0
2,Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.,0,0,0,1,0,0


In [51]:

# ADIM 8.20: Hesap tehdidi V3

import re

def hesap_tehdidi_v3(mesaj):

    metin = turkce_ascii(mesaj)

    hesap_kaliplari = [
        r"\bhesab\w*",
        r"\bhesap\w*",
        r"\berisim\w*",
        r"\boturum\w*"
    ]

    tehdit_kaliplari = [
        r"kapatil\w*",
        r"askiya\s+alin\w*",
        r"engellen\w*",
        r"bloke\s+edil\w*",
        r"bloke\s+ol\w*",
        r"durdurul\w*",
        r"kilitlen\w*"
    ]

    hesaplar = [
        esleme
        for kalip in hesap_kaliplari
        for esleme in re.finditer(kalip, metin)
    ]

    tehditler = [
        esleme
        for kalip in tehdit_kaliplari
        for esleme in re.finditer(kalip, metin)
    ]

    for hesap in hesaplar:
        for tehdit in tehditler:

            mesafe = max(
                hesap.start() - tehdit.end(),
                tehdit.start() - hesap.end(),
                0
            )

            if mesafe <= 60:
                return 1

    return 0


# Kontrol testleri
testler = [
    "Hesabiniz bloke olacak, bilgilerinizi guncelleyin.",
    "Hesabınız bloke edilecek.",
    "Google hesabınız askıya alınacak.",
    "Hesabınız kapatılacak.",
    "Hesabınız başarıyla oluşturuldu.",
    "Yarın toplantımız var."
]

for mesaj in testler:
    print(mesaj, "→", hesap_tehdidi_v3(mesaj))


Hesabiniz bloke olacak, bilgilerinizi guncelleyin. → 1
Hesabınız bloke edilecek. → 1
Google hesabınız askıya alınacak. → 1
Hesabınız kapatılacak. → 1
Hesabınız başarıyla oluşturuldu. → 0
Yarın toplantımız var. → 0


In [52]:

# ADIM 8.21: Hassas bilgi talebi V3

import re

def bilgi_talebi_v3(mesaj):
    metin = turkce_ascii(mesaj)

    # Hassas bilgi ifadeleri
    hassas_kaliplar = [
        r"\bsifreniz\w*",
        r"\bdogrulama\s+kodun\w*",
        r"\bkimlik\s+(?:numaraniz|bilgileriniz)\w*",
        r"\bkart\s+bilgileriniz\w*",
        r"\bbilgileriniz\w*"
    ]

    # Talep fiilleri
    fiil_kaliplari = [
        r"\bgonder\w*",
        r"\bpaylas\w*",
        r"\bilet\w*",
        r"\bgir\w*",
        r"\byaz\w*",
        r"\bguncelle\w*"
    ]

    # Olumsuz emirleri ayrı değerlendir
    olumsuz_kalip = re.compile(
        r"^(?:gonderme|paylasma|iletme|girme|"
        r"yazma|guncelleme)"
        r"(?:yin|yiniz|n|niz)?$"
    )

    kelimeler = re.findall(r"\b\w+\b", metin)

    # Hassas bilgi var mı?
    hassas_var = any(
        re.search(k, metin) for k in hassas_kaliplar
    )

    if not hassas_var:
        return 0

    # Olumsuz fiilleri ele
    for kelime in kelimeler:
        if olumsuz_kalip.fullmatch(kelime):
            continue

        if any(re.fullmatch(k, kelime) for k in fiil_kaliplari):
            return 1

    return 0


testler = [
    "Hesabiniz bloke olacak, bilgilerinizi guncelleyin.",
    "Kimlik bilgilerinizi hemen bu numaraya iletin.",
    "Şifrenizi gönderin.",
    "Kart bilgilerinizi sisteme giriniz.",
    "Şifrenizi kimseyle paylaşmayın.",
    "Sifrenizi asla kimseye gondermeyin.",
    "Bilgilerinizi güncelleyin.",
    "Yarın toplantımız var."
]

for mesaj in testler:
    print(mesaj, "→", bilgi_talebi_v3(mesaj))


Hesabiniz bloke olacak, bilgilerinizi guncelleyin. → 1
Kimlik bilgilerinizi hemen bu numaraya iletin. → 1
Şifrenizi gönderin. → 1
Kart bilgilerinizi sisteme giriniz. → 1
Şifrenizi kimseyle paylaşmayın. → 0
Sifrenizi asla kimseye gondermeyin. → 0
Bilgilerinizi güncelleyin. → 1
Yarın toplantımız var. → 0


In [53]:

# ADIM 8.22: Ödül ve ödeme talebi analizi

import re

def odul_odeme_analizi(mesaj):

    # Türkçe karakterleri standartlaştır
    metin = turkce_ascii(mesaj)

    # Ödül ve hediye vaatleri
    odul_kaliplari = [
        r"\bodul\w*.{0,35}kazan\w*",
        r"\bhediye\w*.{0,35}kazan\w*",
        r"\bcekilis\w*.{0,35}kazan\w*",
        r"\bodulunuzu\s+al\w*",
        r"\bhediyenizi\s+al\w*"
    ]

    # Ödeme ve ücret talepleri
    odeme_kaliplari = [
        r"\bislem\s+ucret\w*",
        r"\bkargo\s+bedel\w*",
        r"\bodeme\s+yap\w*",
        r"\bpara\s+gonder\w*",
        r"\bucret\s+ode\w*",
        r"\b\d+\s*(?:tl|₺)\b"
    ]

    odul_var = int(any(
        re.search(k, metin) for k in odul_kaliplari
    ))

    odeme_var = int(any(
        re.search(k, metin) for k in odeme_kaliplari
    ))

    return {
        "odul_vaadi": odul_var,
        "odeme_talebi": odeme_var,
        "odul_odeme_birlikte": int(
            odul_var == 1 and odeme_var == 1
        )
    }


# Kontrol testleri
testler = [
    "Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.",
    "Tebrikler, hediye kazandınız! Kargo bedelini ödeyin.",
    "Ödülünüzü almak için 250 TL gönderin.",
    "Yarın okulda ödül töreni yapılacak.",
    "Elektrik faturanız için ödeme yapın.",
    "Tebrikler, yarışmada ödül kazandınız!",
    "Yarın saat 10'da toplantımız var."
]

for mesaj in testler:
    print("\nMesaj:", mesaj)
    print(odul_odeme_analizi(mesaj))



Mesaj: Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.
{'odul_vaadi': 1, 'odeme_talebi': 1, 'odul_odeme_birlikte': 1}

Mesaj: Tebrikler, hediye kazandınız! Kargo bedelini ödeyin.
{'odul_vaadi': 1, 'odeme_talebi': 1, 'odul_odeme_birlikte': 1}

Mesaj: Ödülünüzü almak için 250 TL gönderin.
{'odul_vaadi': 1, 'odeme_talebi': 1, 'odul_odeme_birlikte': 1}

Mesaj: Yarın okulda ödül töreni yapılacak.
{'odul_vaadi': 0, 'odeme_talebi': 0, 'odul_odeme_birlikte': 0}

Mesaj: Elektrik faturanız için ödeme yapın.
{'odul_vaadi': 0, 'odeme_talebi': 1, 'odul_odeme_birlikte': 0}

Mesaj: Tebrikler, yarışmada ödül kazandınız!
{'odul_vaadi': 1, 'odeme_talebi': 0, 'odul_odeme_birlikte': 0}

Mesaj: Yarın saat 10'da toplantımız var.
{'odul_vaadi': 0, 'odeme_talebi': 0, 'odul_odeme_birlikte': 0}


In [54]:

# ADIM 8.23: SiberDil V3 birleşik analiz

def siberdil_analiz_v3(mesaj):

    # Önce mevcut analiz sistemini çalıştır
    analiz = siberdil_analiz_v2(mesaj)

    # Geliştirilmiş Türkçe içerik analizleri
    analiz["hesap_tehdidi"] = hesap_tehdidi_v3(mesaj)
    analiz["bilgi_talebi"] = bilgi_talebi_v3(mesaj)

    # Ödül ve ödeme analizi
    odeme_sonuclari = odul_odeme_analizi(mesaj)
    analiz.update(odeme_sonuclari)

    return analiz


# Birleşik sistemi test et
mesaj = "Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor."

sonuc = siberdil_analiz_v3(mesaj)

print("SİBERDİL V3 ANALİZ SONUCU")
print("-" * 35)

for ozellik, deger in sonuc.items():
    print(f"{ozellik}: {deger}")


SİBERDİL V3 ANALİZ SONUCU
-----------------------------------
mesaj: Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.
url_var: 0
url_sayisi: 0
taklit_suphesi: 0
karakter_eksikligi: 0
diger_taninmayan: 0
diger_taninmayan_orani: 0.0
aciliyet: 0
hesap_tehdidi: 0
odul_vaadi: 1
bilgi_talebi: 0
risk_puani: 10
risk_seviyesi: Düşük
odeme_talebi: 1
odul_odeme_birlikte: 1


In [55]:

# ADIM 8.24: SiberDil V4 risk puanlama

def risk_puani_hesapla_v4(analiz):

    # Mevcut V3 risk puanı
    puan = risk_puani_hesapla_v3(analiz)

    # Ödül vaadi + ödeme talebi birleşimi
    if analiz.get("odul_odeme_birlikte", 0) == 1:
        puan += 45

    return min(100, puan)


def siberdil_analiz_v4(mesaj):

    # Tüm özellikleri hesapla
    analiz = siberdil_analiz_v3(mesaj)

    # Risk puanını güncelle
    puan = risk_puani_hesapla_v4(analiz)

    analiz["risk_puani"] = puan
    analiz["risk_seviyesi"] = risk_seviyesi_belirle(puan)

    return analiz


# İlk kontrol
mesaj = "Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor."

sonuc = siberdil_analiz_v4(mesaj)

print("SİBERDİL V4")
print("Mesaj:", mesaj)
print("Ödül vaadi:", sonuc["odul_vaadi"])
print("Ödeme talebi:", sonuc["odeme_talebi"])
print("Ödül + ödeme:", sonuc["odul_odeme_birlikte"])
print("Risk puanı:", sonuc["risk_puani"])
print("Risk seviyesi:", sonuc["risk_seviyesi"])


SİBERDİL V4
Mesaj: Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.
Ödül vaadi: 1
Ödeme talebi: 1
Ödül + ödeme: 1
Risk puanı: 55
Risk seviyesi: Orta


In [56]:

# ADIM 8.25: SiberDil V4 karşılaştırmalı test

test_mesajlari = [
    ("Yarın saat 10'da toplantımız var.", 0),
    ("Elektrik faturanız için ödeme yapın.", 0),
    ("Tebrikler, yarışmada ödül kazandınız!", 0),
    ("Lütfen şifrenizi kimseyle paylaşmayın.", 0),
    ("Kargonuz dağıtıma çıkarılmıştır.", 0),
    ("Okul gezisi için 250 TL yatırmanız gerekiyor.", 0),

    ("Hesabiniz bloke olacak, bilgilerinizi guncelleyin.", 1),
    ("Kimlik bilgilerinizi hemen bu numaraya iletin.", 1),
    ("Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.", 1),
    ("Tebrikler, hediye kazandınız! Kargo bedelini ödeyin.", 1),
    ("Şifrenizi hemen gönderin, yoksa hesabınız kapanacak.", 1),
    ("Hesabınız askıya alınacak. Şifrenizi paylaşın.", 1)
]

dogru = 0
yanlis_pozitif = 0
yanlis_negatif = 0

for mesaj, gercek in test_mesajlari:

    sonuc = siberdil_analiz_v4(mesaj)
    puan = sonuc["risk_puani"]

    # Orta ve yüksek riskleri şüpheli kabul et
    tahmin = int(puan >= 30)

    if tahmin == gercek:
        dogru += 1
    elif tahmin == 1:
        yanlis_pozitif += 1
    else:
        yanlis_negatif += 1

    print(
        f"Gerçek: {gercek} | "
        f"Tahmin: {tahmin} | "
        f"Puan: {puan:3} | {mesaj}"
    )

print("\n--- TEST ÖZETİ ---")
print("Doğru:", dogru, "/", len(test_mesajlari))
print("Doğruluk:", round(dogru / len(test_mesajlari) * 100, 1), "%")
print("Yanlış pozitif:", yanlis_pozitif)
print("Yanlış negatif:", yanlis_negatif)


Gerçek: 0 | Tahmin: 0 | Puan:   0 | Yarın saat 10'da toplantımız var.
Gerçek: 0 | Tahmin: 0 | Puan:   0 | Elektrik faturanız için ödeme yapın.
Gerçek: 0 | Tahmin: 0 | Puan:  10 | Tebrikler, yarışmada ödül kazandınız!
Gerçek: 0 | Tahmin: 0 | Puan:   0 | Lütfen şifrenizi kimseyle paylaşmayın.
Gerçek: 0 | Tahmin: 0 | Puan:   0 | Kargonuz dağıtıma çıkarılmıştır.
Gerçek: 0 | Tahmin: 0 | Puan:   0 | Okul gezisi için 250 TL yatırmanız gerekiyor.
Gerçek: 1 | Tahmin: 1 | Puan:  70 | Hesabiniz bloke olacak, bilgilerinizi guncelleyin.
Gerçek: 1 | Tahmin: 1 | Puan:  55 | Kimlik bilgilerinizi hemen bu numaraya iletin.
Gerçek: 1 | Tahmin: 1 | Puan:  55 | Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.
Gerçek: 1 | Tahmin: 1 | Puan:  55 | Tebrikler, hediye kazandınız! Kargo bedelini ödeyin.
Gerçek: 1 | Tahmin: 1 | Puan:  55 | Şifrenizi hemen gönderin, yoksa hesabınız kapanacak.
Gerçek: 1 | Tahmin: 1 | Puan:  65 | Hesabınız askıya alınacak. Şifrenizi paylaşın.

--- TEST ÖZETİ ---
Doğru: 12 / 12
Doğru

In [57]:

# ADIM 8.26: Zorlayıcı mesaj testi

zor_testler = [
    ("Hesabınız kilitlendi, şifrenizi yenilemek için bağlantıya tıklayın.", 1),
    ("Bankanızdan arıyoruz. Kart numaranızı bizimle paylaşın.", 1),
    ("Kargonuz teslim edilemedi. Adresinizi doğrulayın.", 1),
    ("Tebrikler! 5000 TL ödülünüzü almak için vergi yatırın.", 1),
    ("Sayın müşterimiz, hesabınızın güvenliği için PIN kodunuzu iletin.", 1),
    ("Sifrenizi hemen gonderin yoksa hesabinizi kapatacagiz.", 1),
    ("Paketinizi almak için https://kargo-onay.example adresine girin.", 1),
    ("Hesabınızda şüpheli işlem var. Kimlik bilgilerinizi paylaşın.", 1),

    ("Bankanızın şifrenizi istemeyeceğini unutmayın.", 0),
    ("Kargonuz yarın teslim edilecek.", 0),
    ("Ödül töreninde 5000 TL para ödülü verilecek.", 0),
    ("Kart bilgilerinizi kimseyle paylaşmayın.", 0),
    ("Hesabınızın şifresini düzenli olarak değiştirin.", 0),
    ("Okul gezisi için ödeme yapmanız gerekiyor.", 0),
    ("Yarışmayı kazandınız, tebrikler!", 0),
    ("Hesabınızdan yapılan işlemleri uygulamadan kontrol edin.", 0)
]

dogru = 0
yanlis_pozitif = 0
yanlis_negatif = 0

for mesaj, gercek in zor_testler:
    sonuc = siberdil_analiz_v4(mesaj)
    puan = sonuc["risk_puani"]
    tahmin = int(puan >= 30)

    if tahmin == gercek:
        dogru += 1
    elif tahmin == 1:
        yanlis_pozitif += 1
    else:
        yanlis_negatif += 1

    durum = "DOGRU" if tahmin == gercek else "HATA"

    print(
        f"{durum:5} | Gerçek:{gercek} "
        f"Tahmin:{tahmin} Puan:{puan:3} | {mesaj}"
    )

print("\n--- ZOR TEST ÖZETİ ---")
print("Doğru:", dogru, "/", len(zor_testler))
print("Doğruluk:", round(100 * dogru / len(zor_testler), 1), "%")
print("Yanlış pozitif:", yanlis_pozitif)
print("Yanlış negatif:", yanlis_negatif)


HATA  | Gerçek:1 Tahmin:0 Puan: 15 | Hesabınız kilitlendi, şifrenizi yenilemek için bağlantıya tıklayın.
HATA  | Gerçek:1 Tahmin:0 Puan:  0 | Bankanızdan arıyoruz. Kart numaranızı bizimle paylaşın.
HATA  | Gerçek:1 Tahmin:0 Puan:  0 | Kargonuz teslim edilemedi. Adresinizi doğrulayın.
DOGRU | Gerçek:1 Tahmin:1 Puan: 55 | Tebrikler! 5000 TL ödülünüzü almak için vergi yatırın.
HATA  | Gerçek:1 Tahmin:0 Puan:  5 | Sayın müşterimiz, hesabınızın güvenliği için PIN kodunuzu iletin.
DOGRU | Gerçek:1 Tahmin:1 Puan: 60 | Sifrenizi hemen gonderin yoksa hesabinizi kapatacagiz.
HATA  | Gerçek:1 Tahmin:0 Puan:  0 | Paketinizi almak için https://kargo-onay.example adresine girin.
DOGRU | Gerçek:1 Tahmin:1 Puan: 30 | Hesabınızda şüpheli işlem var. Kimlik bilgilerinizi paylaşın.
DOGRU | Gerçek:0 Tahmin:0 Puan:  0 | Bankanızın şifrenizi istemeyeceğini unutmayın.
DOGRU | Gerçek:0 Tahmin:0 Puan:  0 | Kargonuz yarın teslim edilecek.
DOGRU | Gerçek:0 Tahmin:0 Puan:  0 | Ödül töreninde 5000 TL para ödülü ver

In [58]:

# ADIM 8.27: Hesap tehdidi V4

import re

def hesap_tehdidi_v4(mesaj):

    # Önce mevcut V3 analizini kullan
    if hesap_tehdidi_v3(mesaj) == 1:
        return 1

    metin = turkce_ascii(mesaj)

    yeni_kaliplar = [
        r"\bhesab\w*.{0,45}\bkilitlen\w*",
        r"\bhesab\w*.{0,45}\bkilitli\w*",
        r"\bhesab\w*.{0,45}\bdondurul\w*",
        r"\bhesab\w*.{0,45}\bdevre\s+disi\b"
    ]

    return int(any(
        re.search(kalip, metin)
        for kalip in yeni_kaliplar
    ))


# Kontrol testleri
testler = [
    ("Hesabınız kilitlendi, şifrenizi yenileyin.", 1),
    ("Hesabınız güvenlik nedeniyle kilitlenecek.", 1),
    ("Hesabınız donduruldu.", 1),
    ("Hesabınız devre dışı bırakıldı.", 1),
    ("Hesabınız aktif durumda.", 0),
    ("Kapının kilidi bozuldu.", 0)
]

for mesaj, beklenen in testler:
    sonuc = hesap_tehdidi_v4(mesaj)

    print(
        "DOGRU" if sonuc == beklenen else "HATA",
        "| Beklenen:", beklenen,
        "| Sonuç:", sonuc,
        "|", mesaj
    )



DOGRU | Beklenen: 1 | Sonuç: 1 | Hesabınız kilitlendi, şifrenizi yenileyin.
DOGRU | Beklenen: 1 | Sonuç: 1 | Hesabınız güvenlik nedeniyle kilitlenecek.
DOGRU | Beklenen: 1 | Sonuç: 1 | Hesabınız donduruldu.
DOGRU | Beklenen: 1 | Sonuç: 1 | Hesabınız devre dışı bırakıldı.
DOGRU | Beklenen: 0 | Sonuç: 0 | Hesabınız aktif durumda.
DOGRU | Beklenen: 0 | Sonuç: 0 | Kapının kilidi bozuldu.


In [59]:

# ADIM 8.28: Hassas bilgi talebi V4

import re

def bilgi_talebi_v4(mesaj):

    # Önceki analizde tespit edildiyse koru
    if bilgi_talebi_v3(mesaj) == 1:
        return 1

    metin = turkce_ascii(mesaj)

    hassas_kaliplar = [
        r"\bkart\s+numaraniz\w*",
        r"\bpin\s+kodun\w*",
        r"\bcvv\s+kodun\w*",
        r"\bkart\s+sifreniz\w*"
    ]

    talep_kaliplari = [
        r"\bpaylas\w*",
        r"\bilet\w*",
        r"\bgonder\w*",
        r"\bgir\w*",
        r"\byaz\w*"
    ]

    olumsuz_kalip = re.compile(
        r"^(?:paylasma|iletme|gonderme|"
        r"girme|yazma)(?:yin|yiniz|n|niz)?$"
    )

    hassas_var = any(
        re.search(k, metin) for k in hassas_kaliplar
    )

    if not hassas_var:
        return 0

    kelimeler = re.findall(r"\b\w+\b", metin)

    for kelime in kelimeler:
        if olumsuz_kalip.fullmatch(kelime):
            continue

        if any(re.fullmatch(k, kelime) for k in talep_kaliplari):
            return 1

    return 0


testler = [
    ("Kart numaranızı bizimle paylaşın.", 1),
    ("PIN kodunuzu hemen iletin.", 1),
    ("CVV kodunuzu gönderin.", 1),
    ("Kart şifrenizi sisteme girin.", 1),
    ("Kart numaranızı kimseyle paylaşmayın.", 0),
    ("PIN kodunuzu asla göndermeyin.", 0),
    ("Kartımın son kullanma tarihi yaklaşıyor.", 0),
    ("Yarın toplantımız var.", 0)
]

for mesaj, beklenen in testler:
    sonuc = bilgi_talebi_v4(mesaj)
    durum = "DOGRU" if sonuc == beklenen else "HATA"
    print(durum, "|", sonuc, "|", mesaj)


DOGRU | 1 | Kart numaranızı bizimle paylaşın.
DOGRU | 1 | PIN kodunuzu hemen iletin.
DOGRU | 1 | CVV kodunuzu gönderin.
DOGRU | 1 | Kart şifrenizi sisteme girin.
DOGRU | 0 | Kart numaranızı kimseyle paylaşmayın.
DOGRU | 0 | PIN kodunuzu asla göndermeyin.
DOGRU | 0 | Kartımın son kullanma tarihi yaklaşıyor.
DOGRU | 0 | Yarın toplantımız var.


In [60]:

# ADIM 8.29: Kargo ve adres doğrulama analizi

import re

def kargo_adres_analizi(mesaj):

    metin = turkce_ascii(mesaj)

    teslimat_sorunu_kaliplari = [
        r"\bteslim\s+edileme\w*",
        r"\bteslimat\s+basarisiz\b",
        r"\bkargo\w*.{0,40}\bbekletil\w*",
        r"\bpaket\w*.{0,40}\biade\s+edil\w*"
    ]

    adres_talebi_kaliplari = [
        r"\badres\w*.{0,35}\bdogrula\w*",
        r"\badres\w*.{0,35}\bguncelle\w*",
        r"\badres\w*.{0,35}\bgir\w*"
    ]

    sorun_var = int(any(
        re.search(k, metin)
        for k in teslimat_sorunu_kaliplari
    ))

    adres_talebi = int(any(
        re.search(k, metin)
        for k in adres_talebi_kaliplari
    ))

    return {
        "teslimat_sorunu": sorun_var,
        "adres_talebi": adres_talebi,
        "kargo_adres_riski": int(
            sorun_var == 1 and adres_talebi == 1
        )
    }


# Kontrol testleri
testler = [
    ("Kargonuz teslim edilemedi. Adresinizi doğrulayın.", 1),
    ("Paketiniz teslim edilemedi, adresinizi güncelleyin.", 1),
    ("Teslimat başarısız. Adresinizi doğrulayın.", 1),
    ("Kargonuz yarın teslim edilecek.", 0),
    ("Adresinizi güncellemek için belediyeye başvurun.", 0),
    ("Kargonuz teslim edilemedi, yarın tekrar denenecek.", 0),
    ("Yarın okulda toplantımız var.", 0)
]

for mesaj, beklenen in testler:

    sonuc = kargo_adres_analizi(mesaj)
    tahmin = sonuc["kargo_adres_riski"]

    durum = "DOGRU" if tahmin == beklenen else "HATA"

    print(durum, "|", tahmin, "|", mesaj)


DOGRU | 1 | Kargonuz teslim edilemedi. Adresinizi doğrulayın.
DOGRU | 1 | Paketiniz teslim edilemedi, adresinizi güncelleyin.
DOGRU | 1 | Teslimat başarısız. Adresinizi doğrulayın.
DOGRU | 0 | Kargonuz yarın teslim edilecek.
DOGRU | 0 | Adresinizi güncellemek için belediyeye başvurun.
DOGRU | 0 | Kargonuz teslim edilemedi, yarın tekrar denenecek.
DOGRU | 0 | Yarın okulda toplantımız var.


In [61]:

# ADIM 8.30: SiberDil V5 birleşik analiz

def siberdil_analiz_v5(mesaj):

    # Mevcut V4 analizini çalıştır
    analiz = siberdil_analiz_v4(mesaj)

    # Yeni dilsel analizleri uygula
    analiz["hesap_tehdidi"] = hesap_tehdidi_v4(mesaj)
    analiz["bilgi_talebi"] = bilgi_talebi_v4(mesaj)

    # Kargo ve adres analizi
    kargo_sonucu = kargo_adres_analizi(mesaj)
    analiz.update(kargo_sonucu)

    # Risk puanını güncel özelliklerle tekrar hesapla
    puan = risk_puani_hesapla_v4(analiz)

    analiz["risk_puani"] = puan
    analiz["risk_seviyesi"] = risk_seviyesi_belirle(puan)

    return analiz


# Yeni özelliklerin ana sisteme aktarıldığını kontrol et
testler = [
    "Hesabınız kilitlendi, şifrenizi yenileyin.",
    "Kart numaranızı bizimle paylaşın.",
    "PIN kodunuzu hemen iletin.",
    "Kargonuz teslim edilemedi. Adresinizi doğrulayın."
]

for mesaj in testler:

    sonuc = siberdil_analiz_v5(mesaj)

    print("\nMESAJ:", mesaj)
    print("Hesap tehdidi:", sonuc["hesap_tehdidi"])
    print("Bilgi talebi:", sonuc["bilgi_talebi"])
    print("Kargo-adres riski:", sonuc["kargo_adres_riski"])
    print("Risk puanı:", sonuc["risk_puani"])
    print("Risk seviyesi:", sonuc["risk_seviyesi"])



MESAJ: Hesabınız kilitlendi, şifrenizi yenileyin.
Hesap tehdidi: 1
Bilgi talebi: 0
Kargo-adres riski: 0
Risk puanı: 15
Risk seviyesi: Düşük

MESAJ: Kart numaranızı bizimle paylaşın.
Hesap tehdidi: 0
Bilgi talebi: 1
Kargo-adres riski: 0
Risk puanı: 30
Risk seviyesi: Orta

MESAJ: PIN kodunuzu hemen iletin.
Hesap tehdidi: 0
Bilgi talebi: 1
Kargo-adres riski: 0
Risk puanı: 60
Risk seviyesi: Yüksek

MESAJ: Kargonuz teslim edilemedi. Adresinizi doğrulayın.
Hesap tehdidi: 0
Bilgi talebi: 0
Kargo-adres riski: 1
Risk puanı: 0
Risk seviyesi: Düşük


In [62]:

# ADIM 8.31: SiberDil V6 risk puanlama

def risk_puani_hesapla_v5(analiz):

    # Mevcut puanlama sistemini koru
    puan = risk_puani_hesapla_v4(analiz)

    # Teslimat sorunu + adres doğrulama isteği
    if analiz.get("kargo_adres_riski", 0) == 1:
        puan += 35

    return min(100, puan)


def siberdil_analiz_v6(mesaj):

    analiz = siberdil_analiz_v5(mesaj)

    puan = risk_puani_hesapla_v5(analiz)

    analiz["risk_puani"] = puan
    analiz["risk_seviyesi"] = risk_seviyesi_belirle(puan)

    return analiz


# Kontrol testleri
testler = [
    "Kargonuz teslim edilemedi. Adresinizi doğrulayın.",
    "Kargonuz yarın teslim edilecek.",
    "Adresinizi güncellemek için belediyeye başvurun.",
    "Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor."
]

for mesaj in testler:

    sonuc = siberdil_analiz_v6(mesaj)

    print("\nMesaj:", mesaj)
    print("Kargo-adres riski:", sonuc["kargo_adres_riski"])
    print("Risk puanı:", sonuc["risk_puani"])
    print("Risk seviyesi:", sonuc["risk_seviyesi"])



Mesaj: Kargonuz teslim edilemedi. Adresinizi doğrulayın.
Kargo-adres riski: 1
Risk puanı: 35
Risk seviyesi: Orta

Mesaj: Kargonuz yarın teslim edilecek.
Kargo-adres riski: 0
Risk puanı: 0
Risk seviyesi: Düşük

Mesaj: Adresinizi güncellemek için belediyeye başvurun.
Kargo-adres riski: 0
Risk puanı: 0
Risk seviyesi: Düşük

Mesaj: Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.
Kargo-adres riski: 0
Risk puanı: 55
Risk seviyesi: Orta


In [63]:

# ADIM 8.32: Bağlantıya yönlendirme analizi

import re

def baglanti_yonlendirme_analizi(mesaj):

    metin = turkce_ascii(mesaj)

    kaliplar = [
        r"\bbaglantiya\s+tikla\w*",
        r"\blinke\s+tikla\w*",
        r"\blink\w*\s+ac\w*",
        r"\bbaglantiyi\s+ac\w*",
        r"\blinki\s+ac\w*",
        r"\basagidaki\s+baglanti\w*.{0,30}\btikla\w*"
    ]

    return int(any(
        re.search(kalip, metin)
        for kalip in kaliplar
    ))


# Kontrol testleri
testler = [
    ("Hesabınız kilitlendi, bağlantıya tıklayın.", 1),
    ("Şifrenizi yenilemek için linke tıklayın.", 1),
    ("Aşağıdaki bağlantıya hemen tıklayın.", 1),
    ("Gönderdiğimiz linki açın.", 1),
    ("Ders materyali için bağlantıya tıklayın.", 1),
    ("Şüpheli bağlantılara tıklamayın.", 0),
    ("Bağlantıyı açmayın.", 0),
    ("Yarın okulda toplantımız var.", 0)
]

for mesaj, beklenen in testler:

    sonuc = baglanti_yonlendirme_analizi(mesaj)

    durum = "DOGRU" if sonuc == beklenen else "HATA"

    print(
        durum,
        "| Beklenen:", beklenen,
        "| Sonuç:", sonuc,
        "|", mesaj
    )


DOGRU | Beklenen: 1 | Sonuç: 1 | Hesabınız kilitlendi, bağlantıya tıklayın.
DOGRU | Beklenen: 1 | Sonuç: 1 | Şifrenizi yenilemek için linke tıklayın.
DOGRU | Beklenen: 1 | Sonuç: 1 | Aşağıdaki bağlantıya hemen tıklayın.
DOGRU | Beklenen: 1 | Sonuç: 1 | Gönderdiğimiz linki açın.
DOGRU | Beklenen: 1 | Sonuç: 1 | Ders materyali için bağlantıya tıklayın.
DOGRU | Beklenen: 0 | Sonuç: 0 | Şüpheli bağlantılara tıklamayın.
HATA | Beklenen: 0 | Sonuç: 1 | Bağlantıyı açmayın.
DOGRU | Beklenen: 0 | Sonuç: 0 | Yarın okulda toplantımız var.


In [64]:

# ADIM 8.33: Bağlantıya yönlendirme V2
# Olumsuz emirleri ayırt etme

import re

def baglanti_yonlendirme_v2(mesaj):

    metin = turkce_ascii(mesaj)

    # Önce olumsuz emirleri belirle
    olumsuz_kaliplar = [
        r"\btiklama\w*",
        r"\bacma\w*"
    ]

    # Olumsuz emirleri tespit öncesinde kaldır
    temiz_metin = metin

    for kalip in olumsuz_kaliplar:
        temiz_metin = re.sub(kalip, " ", temiz_metin)

    # Mevcut yönlendirme analizini uygula
    return baglanti_yonlendirme_analizi(temiz_metin)


testler = [
    ("Hesabınız kilitlendi, bağlantıya tıklayın.", 1),
    ("Şifrenizi yenilemek için linke tıklayın.", 1),
    ("Aşağıdaki bağlantıya hemen tıklayın.", 1),
    ("Gönderdiğimiz linki açın.", 1),
    ("Ders materyali için bağlantıya tıklayın.", 1),
    ("Şüpheli bağlantılara tıklamayın.", 0),
    ("Bağlantıyı açmayın.", 0),
    ("Yarın okulda toplantımız var.", 0)
]

for mesaj, beklenen in testler:

    sonuc = baglanti_yonlendirme_v2(mesaj)

    durum = "DOGRU" if sonuc == beklenen else "HATA"

    print(durum, "|", sonuc, "|", mesaj)


DOGRU | 1 | Hesabınız kilitlendi, bağlantıya tıklayın.
DOGRU | 1 | Şifrenizi yenilemek için linke tıklayın.
DOGRU | 1 | Aşağıdaki bağlantıya hemen tıklayın.
DOGRU | 1 | Gönderdiğimiz linki açın.
DOGRU | 1 | Ders materyali için bağlantıya tıklayın.
DOGRU | 0 | Şüpheli bağlantılara tıklamayın.
DOGRU | 0 | Bağlantıyı açmayın.
DOGRU | 0 | Yarın okulda toplantımız var.


In [65]:

# ADIM 8.34: SiberDil V7

def siberdil_analiz_v7(mesaj):

    # Mevcut V6 analizini çalıştır
    analiz = siberdil_analiz_v6(mesaj)

    # Bağlantıya yönlendirme özelliğini ekle
    analiz["baglanti_yonlendirme"] = (
        baglanti_yonlendirme_v2(mesaj)
    )

    # Önceki puanı al
    puan = analiz["risk_puani"]

    # Hesap tehdidi + bağlantıya yönlendirme
    if (
        analiz["hesap_tehdidi"] == 1
        and analiz["baglanti_yonlendirme"] == 1
    ):
        puan += 25

    # Puan 100'ü geçmesin
    puan = min(100, puan)

    analiz["risk_puani"] = puan
    analiz["risk_seviyesi"] = risk_seviyesi_belirle(puan)

    return analiz


# Kontrol testleri
testler = [
    ("Hesabınız kilitlendi, bağlantıya tıklayın.", 40),
    ("Ders materyali için bağlantıya tıklayın.", 0),
    ("Şüpheli bağlantılara tıklamayın.", 0),
    ("Hesabınız kilitlendi.", 15)
]

for mesaj, beklenen in testler:

    sonuc = siberdil_analiz_v7(mesaj)
    puan = sonuc["risk_puani"]

    durum = "DOGRU" if puan == beklenen else "HATA"

    print(
        durum,
        "| Beklenen:", beklenen,
        "| Puan:", puan,
        "|", mesaj
    )


DOGRU | Beklenen: 40 | Puan: 40 | Hesabınız kilitlendi, bağlantıya tıklayın.
DOGRU | Beklenen: 0 | Puan: 0 | Ders materyali için bağlantıya tıklayın.
DOGRU | Beklenen: 0 | Puan: 0 | Şüpheli bağlantılara tıklamayın.
DOGRU | Beklenen: 15 | Puan: 15 | Hesabınız kilitlendi.


In [66]:

# ADIM 8.35: SiberDil V4 ve V7 karşılaştırması

# Adım 8.26'daki zor_testler listesini kullanıyoruz

def performans_testi(analiz_fonksiyonu, test_verileri):

    dogru = 0
    yanlis_pozitif = 0
    yanlis_negatif = 0

    for mesaj, gercek in test_verileri:

        sonuc = analiz_fonksiyonu(mesaj)
        puan = sonuc["risk_puani"]
        tahmin = int(puan >= 30)

        if tahmin == gercek:
            dogru += 1
        elif tahmin == 1:
            yanlis_pozitif += 1
        else:
            yanlis_negatif += 1

        durum = "DOGRU" if tahmin == gercek else "HATA"

        print(
            f"{durum:5} | "
            f"Gerçek:{gercek} "
            f"Tahmin:{tahmin} "
            f"Puan:{puan:3} | {mesaj}"
        )

    toplam = len(test_verileri)

    return {
        "dogru": dogru,
        "toplam": toplam,
        "dogruluk": round(100 * dogru / toplam, 1),
        "yanlis_pozitif": yanlis_pozitif,
        "yanlis_negatif": yanlis_negatif
    }


print("===== SİBERDİL V4 =====")
sonuc_v4 = performans_testi(
    siberdil_analiz_v4, zor_testler
)

print("\n===== SİBERDİL V7 =====")
sonuc_v7 = performans_testi(
    siberdil_analiz_v7, zor_testler
)

print("\n===== KARŞILAŞTIRMA =====")

for alan in [
    "dogru",
    "dogruluk",
    "yanlis_pozitif",
    "yanlis_negatif"
]:
    print(
        alan,
        "| V4:", sonuc_v4[alan],
        "| V7:", sonuc_v7[alan]
    )


===== SİBERDİL V4 =====
HATA  | Gerçek:1 Tahmin:0 Puan: 15 | Hesabınız kilitlendi, şifrenizi yenilemek için bağlantıya tıklayın.
HATA  | Gerçek:1 Tahmin:0 Puan:  0 | Bankanızdan arıyoruz. Kart numaranızı bizimle paylaşın.
HATA  | Gerçek:1 Tahmin:0 Puan:  0 | Kargonuz teslim edilemedi. Adresinizi doğrulayın.
DOGRU | Gerçek:1 Tahmin:1 Puan: 55 | Tebrikler! 5000 TL ödülünüzü almak için vergi yatırın.
HATA  | Gerçek:1 Tahmin:0 Puan:  5 | Sayın müşterimiz, hesabınızın güvenliği için PIN kodunuzu iletin.
DOGRU | Gerçek:1 Tahmin:1 Puan: 60 | Sifrenizi hemen gonderin yoksa hesabinizi kapatacagiz.
HATA  | Gerçek:1 Tahmin:0 Puan:  0 | Paketinizi almak için https://kargo-onay.example adresine girin.
DOGRU | Gerçek:1 Tahmin:1 Puan: 30 | Hesabınızda şüpheli işlem var. Kimlik bilgilerinizi paylaşın.
DOGRU | Gerçek:0 Tahmin:0 Puan:  0 | Bankanızın şifrenizi istemeyeceğini unutmayın.
DOGRU | Gerçek:0 Tahmin:0 Puan:  0 | Kargonuz yarın teslim edilecek.
DOGRU | Gerçek:0 Tahmin:0 Puan:  0 | Ödül törenind

In [67]:

# ADIM 8.36: V7'nin kaçırdığı mesajları bul

for mesaj, gercek in zor_testler:

    sonuc = siberdil_analiz_v7(mesaj)
    tahmin = int(sonuc["risk_puani"] >= 30)

    if tahmin != gercek:
        print("HATALI MESAJ:", mesaj)
        print("Gerçek etiket:", gercek)
        print("Tahmin:", tahmin)
        print("Risk puanı:", sonuc["risk_puani"])
        print("\nAnaliz ayrıntıları:")

        for ozellik, deger in sonuc.items():
            print(ozellik, ":", deger)


HATALI MESAJ: Paketinizi almak için https://kargo-onay.example adresine girin.
Gerçek etiket: 1
Tahmin: 0
Risk puanı: 0

Analiz ayrıntıları:
mesaj : Paketinizi almak için https://kargo-onay.example adresine girin.
url_var : 1
url_sayisi : 1
taklit_suphesi : 0
karakter_eksikligi : 0
diger_taninmayan : 0
diger_taninmayan_orani : 0.0
aciliyet : 0
hesap_tehdidi : 0
odul_vaadi : 0
bilgi_talebi : 0
risk_puani : 0
risk_seviyesi : Düşük
odeme_talebi : 0
odul_odeme_birlikte : 0
teslimat_sorunu : 0
adres_talebi : 1
kargo_adres_riski : 0
baglanti_yonlendirme : 0


In [68]:

# ADIM 8.37: Kargo bağlantısı analizi

import re

def kargo_baglanti_analizi(mesaj):

    metin = turkce_ascii(mesaj)

    url_var = int(len(url_bul(mesaj)) > 0)

    kargo_var = int(bool(re.search(
        r"\b(?:kargo\w*|paket\w*|teslimat\w*)\b",
        metin
    )))

    yonlendirme_kaliplari = [
        r"\bpaket\w*.{0,50}\balmak\s+icin\b",
        r"\bkargo\w*.{0,50}\balmak\s+icin\b",
        r"\bteslimat\w*.{0,50}\btamamla\w*",
        r"\badres\w*.{0,50}\bgir\w*",
        r"\badres\w*.{0,50}\bdogrula\w*"
    ]

    yonlendirme = int(any(
        re.search(k, metin)
        for k in yonlendirme_kaliplari
    ))

    return {
        "kargo_url_var": url_var,
        "kargo_ifadesi": kargo_var,
        "kargo_yonlendirme": yonlendirme,
        "kargo_link_birlikte": int(
            url_var == 1
            and kargo_var == 1
            and yonlendirme == 1
        )
    }


testler = [
    ("Paketinizi almak için https://kargo-onay.example adresine girin.", 1),
    ("Kargonuzu almak için https://teslimat.example adresine girin.", 1),
    ("Kargonuz yarın teslim edilecek.", 0),
    ("Paketiniz yola çıktı: https://takip.example", 0),
    ("Ders notları: https://okul.example", 0),
    ("Adresinizi doğrulayın: https://ornek.example", 0)
]

for mesaj, beklenen in testler:

    sonuc = kargo_baglanti_analizi(mesaj)
    tahmin = sonuc["kargo_link_birlikte"]

    print(
        "DOGRU" if tahmin == beklenen else "HATA",
        "| Beklenen:", beklenen,
        "| Sonuç:", tahmin,
        "|", mesaj
    )


DOGRU | Beklenen: 1 | Sonuç: 1 | Paketinizi almak için https://kargo-onay.example adresine girin.
DOGRU | Beklenen: 1 | Sonuç: 1 | Kargonuzu almak için https://teslimat.example adresine girin.
DOGRU | Beklenen: 0 | Sonuç: 0 | Kargonuz yarın teslim edilecek.
DOGRU | Beklenen: 0 | Sonuç: 0 | Paketiniz yola çıktı: https://takip.example
DOGRU | Beklenen: 0 | Sonuç: 0 | Ders notları: https://okul.example
DOGRU | Beklenen: 0 | Sonuç: 0 | Adresinizi doğrulayın: https://ornek.example


In [69]:

# ADIM 8.38: Kargo bağlantısı yanlış alarm kontrolü

testler = [
    ("Paketinizi almak için https://kargo-onay.example adresine girin.",
     "Şüpheli senaryo"),

    ("Kargonuzu almak için https://teslimat.example adresine girin.",
     "Şüpheli senaryo"),

    ("Kargonuzu teslim almak için https://takip.example adresini kullanın.",
     "Meşru olabilecek mesaj"),

    ("Paketinizin durumunu https://takip.example adresinden öğrenebilirsiniz.",
     "Meşru olabilecek mesaj"),

    ("Kargonuz teslim edilemedi. Adresinizi doğrulayın: https://takip.example",
     "Şüpheli senaryo"),

    ("Kargonuz yarın teslim edilecek. https://takip.example",
     "Meşru olabilecek mesaj"),

    ("Paketinizi almak için şubemize kimliğinizle başvurun.",
     "Meşru olabilecek mesaj"),

    ("Ders materyallerini almak için https://okul.example adresine girin.",
     "Meşru olabilecek mesaj")
]

print("SİBERDİL - KARGO BAĞLANTISI KONTROLÜ")
print("-" * 50)

for mesaj, senaryo in testler:

    sonuc = kargo_baglanti_analizi(mesaj)

    print("\nMesaj:", mesaj)
    print("Senaryo:", senaryo)
    print("URL:", sonuc["kargo_url_var"])
    print("Kargo ifadesi:", sonuc["kargo_ifadesi"])
    print("Yönlendirme:", sonuc["kargo_yonlendirme"])
    print("Birleşik gösterge:", sonuc["kargo_link_birlikte"])


SİBERDİL - KARGO BAĞLANTISI KONTROLÜ
--------------------------------------------------

Mesaj: Paketinizi almak için https://kargo-onay.example adresine girin.
Senaryo: Şüpheli senaryo
URL: 1
Kargo ifadesi: 1
Yönlendirme: 1
Birleşik gösterge: 1

Mesaj: Kargonuzu almak için https://teslimat.example adresine girin.
Senaryo: Şüpheli senaryo
URL: 1
Kargo ifadesi: 1
Yönlendirme: 1
Birleşik gösterge: 1

Mesaj: Kargonuzu teslim almak için https://takip.example adresini kullanın.
Senaryo: Meşru olabilecek mesaj
URL: 1
Kargo ifadesi: 1
Yönlendirme: 1
Birleşik gösterge: 1

Mesaj: Paketinizin durumunu https://takip.example adresinden öğrenebilirsiniz.
Senaryo: Meşru olabilecek mesaj
URL: 1
Kargo ifadesi: 1
Yönlendirme: 0
Birleşik gösterge: 0

Mesaj: Kargonuz teslim edilemedi. Adresinizi doğrulayın: https://takip.example
Senaryo: Şüpheli senaryo
URL: 1
Kargo ifadesi: 1
Yönlendirme: 1
Birleşik gösterge: 1

Mesaj: Kargonuz yarın teslim edilecek. https://takip.example
Senaryo: Meşru olabilecek mesaj

In [70]:

# ADIM 8.39: Kargo bağlantısı ek risk analizi

def kargo_baglanti_risk_v2(mesaj):

    temel = kargo_baglanti_analizi(mesaj)
    kargo = kargo_adres_analizi(mesaj)
    odeme = odul_odeme_analizi(mesaj)

    url_var = temel["kargo_url_var"]
    kargo_var = temel["kargo_ifadesi"]

    # Teslimat sorunu + adres doğrulama + URL
    sorun_adres_url = int(
        url_var == 1
        and kargo_var == 1
        and kargo["teslimat_sorunu"] == 1
        and kargo["adres_talebi"] == 1
    )

    # Kargo + URL + ödeme talebi
    kargo_odeme_url = int(
        url_var == 1
        and kargo_var == 1
        and odeme["odeme_talebi"] == 1
    )

    return {
        "kargo_link_birlikte": temel["kargo_link_birlikte"],
        "sorun_adres_url": sorun_adres_url,
        "kargo_odeme_url": kargo_odeme_url
    }


testler = [
    "Paketinizi almak için https://kargo-onay.example adresine girin.",
    "Kargonuzu teslim almak için https://takip.example adresini kullanın.",
    "Kargonuz teslim edilemedi. Adresinizi doğrulayın: https://takip.example",
    "Kargonuzun işlem ücretini ödemek için https://odeme.example adresine girin.",
    "Kargonuz yarın teslim edilecek: https://takip.example"
]

for i, mesaj in enumerate(testler, 1):
    print("\nMESAJ", i)
    print(kargo_baglanti_risk_v2(mesaj))



MESAJ 1
{'kargo_link_birlikte': 1, 'sorun_adres_url': 0, 'kargo_odeme_url': 0}

MESAJ 2
{'kargo_link_birlikte': 1, 'sorun_adres_url': 0, 'kargo_odeme_url': 0}

MESAJ 3
{'kargo_link_birlikte': 1, 'sorun_adres_url': 1, 'kargo_odeme_url': 0}

MESAJ 4
{'kargo_link_birlikte': 1, 'sorun_adres_url': 0, 'kargo_odeme_url': 1}

MESAJ 5
{'kargo_link_birlikte': 0, 'sorun_adres_url': 0, 'kargo_odeme_url': 0}


In [71]:

# ADIM 8.40: SiberDil V8

def siberdil_analiz_v8(mesaj):

    # Mevcut V7 analizini çalıştır
    analiz = siberdil_analiz_v7(mesaj)

    # Yeni kargo bağlantısı göstergeleri
    kargo_sonuc = kargo_baglanti_risk_v2(mesaj)
    analiz.update(kargo_sonuc)

    # V7 puanını koru
    puan = analiz["risk_puani"]

    # Kargo + URL + ödeme talebi
    if analiz["kargo_odeme_url"] == 1:
        puan += 35

    # Aynı kargo-adres riskine tekrar puan ekleme
    puan = min(100, puan)

    analiz["risk_puani"] = puan
    analiz["risk_seviyesi"] = risk_seviyesi_belirle(puan)

    return analiz


# Kontrol testleri
testler = [
    ("Kargonuzun işlem ücretini ödemek için https://odeme.example adresine girin.", 35),
    ("Kargonuz teslim edilemedi. Adresinizi doğrulayın: https://takip.example", 35),
    ("Kargonuz yarın teslim edilecek: https://takip.example", 0),
    ("Kargonuzu teslim almak için https://takip.example adresini kullanın.", 0),
    ("Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.", 55)
]

for mesaj, beklenen in testler:

    sonuc = siberdil_analiz_v8(mesaj)
    puan = sonuc["risk_puani"]

    durum = "DOGRU" if puan == beklenen else "HATA"

    print(
        durum,
        "| Beklenen:", beklenen,
        "| Puan:", puan,
        "|", mesaj
    )


DOGRU | Beklenen: 35 | Puan: 35 | Kargonuzun işlem ücretini ödemek için https://odeme.example adresine girin.
DOGRU | Beklenen: 35 | Puan: 35 | Kargonuz teslim edilemedi. Adresinizi doğrulayın: https://takip.example
DOGRU | Beklenen: 0 | Puan: 0 | Kargonuz yarın teslim edilecek: https://takip.example
DOGRU | Beklenen: 0 | Puan: 0 | Kargonuzu teslim almak için https://takip.example adresini kullanın.
DOGRU | Beklenen: 55 | Puan: 55 | Ödül kazandınız! İşlem ücreti ödemeniz gerekiyor.


In [72]:

# ADIM 8.41: SiberDil sürüm karşılaştırması

surumler = {
    "V4": siberdil_analiz_v4,
    "V7": siberdil_analiz_v7,
    "V8": siberdil_analiz_v8
}

print("===== SİBERDİL SÜRÜM KARŞILAŞTIRMASI =====")

for surum_adi, analiz_fonksiyonu in surumler.items():

    TP = TN = FP = FN = 0

    for mesaj, gercek in zor_testler:

        sonuc = analiz_fonksiyonu(mesaj)
        tahmin = int(sonuc["risk_puani"] >= 30)

        if gercek == 1 and tahmin == 1:
            TP += 1
        elif gercek == 0 and tahmin == 0:
            TN += 1
        elif gercek == 0 and tahmin == 1:
            FP += 1
        else:
            FN += 1

    toplam = TP + TN + FP + FN
    dogruluk = (TP + TN) / toplam * 100
    duyarlilik = TP / (TP + FN) * 100 if TP + FN else 0
    kesinlik = TP / (TP + FP) * 100 if TP + FP else 0

    print(f"\n{surum_adi}")
    print("Doğruluk:", round(dogruluk, 1), "%")
    print("Oltalama duyarlılığı:", round(duyarlilik, 1), "%")
    print("Kesinlik:", round(kesinlik, 1), "%")
    print("Yanlış pozitif:", FP)
    print("Yanlış negatif:", FN)


print("\n===== V8'İN HATALI SINIFLANDIRMALARI =====")

for mesaj, gercek in zor_testler:

    sonuc = siberdil_analiz_v8(mesaj)
    tahmin = int(sonuc["risk_puani"] >= 30)

    if tahmin != gercek:
        print(
            "Gerçek:", gercek,
            "| Tahmin:", tahmin,
            "| Puan:", sonuc["risk_puani"],
            "|", mesaj
        )


===== SİBERDİL SÜRÜM KARŞILAŞTIRMASI =====

V4
Doğruluk: 68.8 %
Oltalama duyarlılığı: 37.5 %
Kesinlik: 100.0 %
Yanlış pozitif: 0
Yanlış negatif: 5

V7
Doğruluk: 93.8 %
Oltalama duyarlılığı: 87.5 %
Kesinlik: 100.0 %
Yanlış pozitif: 0
Yanlış negatif: 1

V8
Doğruluk: 93.8 %
Oltalama duyarlılığı: 87.5 %
Kesinlik: 100.0 %
Yanlış pozitif: 0
Yanlış negatif: 1

===== V8'İN HATALI SINIFLANDIRMALARI =====
Gerçek: 1 | Tahmin: 0 | Puan: 0 | Paketinizi almak için https://kargo-onay.example adresine girin.


In [73]:

# ADIM 8.42: Bağımsız test veri seti hazırlığı

import pandas as pd

# Test verileri için boş tablo
bagimsiz_test_df = pd.DataFrame(
    columns=["mesaj", "etiket", "kaynak"]
)

# 0 = Meşru mesaj
# 1 = Oltalama mesajı
# Etiketler doğrulanmış örneklere dayanmalı.

print("Bağımsız test tablosu oluşturuldu.")
print("Sütunlar:", bagimsiz_test_df.columns.tolist())
print("Mesaj sayısı:", len(bagimsiz_test_df))

# Mevcut geliştirme mesajlarını ayrıca sakla
gelistirme_df = pd.DataFrame(
    zor_testler,
    columns=["mesaj", "etiket"]
)

print("Geliştirme mesajı sayısı:", len(gelistirme_df))


Bağımsız test tablosu oluşturuldu.
Sütunlar: ['mesaj', 'etiket', 'kaynak']
Mesaj sayısı: 0
Geliştirme mesajı sayısı: 16


In [74]:

# ADIM 8.43: Bağımsız test veri setine mesaj ekleme

def test_mesaji_ekle(mesaj, etiket, kaynak):

    global bagimsiz_test_df

    if etiket not in [0, 1]:
        print("HATA: Etiket 0 veya 1 olmalı.")
        return

    if not isinstance(mesaj, str) or not mesaj.strip():
        print("HATA: Mesaj boş olamaz.")
        return

    mesaj = mesaj.strip()

    # Geliştirme mesajlarında var mı?
    if mesaj in gelistirme_df["mesaj"].values:
        print("HATA: Bu mesaj geliştirme veri setinde var.")
        return

    # Bağımsız test tablosunda var mı?
    if mesaj in bagimsiz_test_df["mesaj"].values:
        print("HATA: Bu mesaj zaten eklenmiş.")
        return

    yeni_satir = pd.DataFrame([{
        "mesaj": mesaj,
        "etiket": etiket,
        "kaynak": kaynak
    }])

    bagimsiz_test_df = pd.concat(
        [bagimsiz_test_df, yeni_satir],
        ignore_index=True
    )

    print("Mesaj eklendi. Toplam:", len(bagimsiz_test_df))


# Fonksiyonu kontrol etmek için geçici örnek
test_mesaji_ekle(
    "Bu yalnızca veri ekleme fonksiyonunu kontrol etmek için yazılmıştır.",
    0,
    "Geçici yazılım testi"
)

print(bagimsiz_test_df)


Mesaj eklendi. Toplam: 1
                                                                  mesaj  \
0  Bu yalnızca veri ekleme fonksiyonunu kontrol etmek için yazılmıştır.   

  etiket                kaynak  
0      0  Geçici yazılım testi  


In [75]:

# ADIM 8.44: Geçici test kaydını kaldırma

bagimsiz_test_df = bagimsiz_test_df[
    bagimsiz_test_df["kaynak"] != "Geçici yazılım testi"
].reset_index(drop=True)

print("Geçici kayıt temizlendi.")
print("Bağımsız test mesajı sayısı:", len(bagimsiz_test_df))
print("Sütunlar:", bagimsiz_test_df.columns.tolist())


Geçici kayıt temizlendi.
Bağımsız test mesajı sayısı: 0
Sütunlar: ['mesaj', 'etiket', 'kaynak']
